# NeoMME — The Complete, Hands-On Tutorial

**A single-tower, multimodal-native, multilingual encoder — from first principles to a working visual-document retrieval system.**

<a href="https://colab.research.google.com/github/smkalle/ai_notebooks/blob/main/notebooks/NeoMME_Complete_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

[NeoMME](https://huggingface.co/blog/Hcompany/neomme) is a family of **260M** and **800M** parameter encoders from
[H Company](https://huggingface.co/Hcompany). It is unusual in a way that matters:

> It reads **text tokens and raw image patches in one bidirectional Transformer**, with **no pretrained
> vision tower** and **no causal language model** anywhere in the stack. The whole thing — patch stem
> included — is trained from scratch with a masked (discrete-diffusion) objective.

That single design decision is what this notebook is really about. Everything downstream — the 32×32 patch
grid, the two-axis M-RoPE positions, the 16,384-token context, the `<doc>`/`<query>` markers, the fact that
one forward pass gives you *both* a dense vector and a bag of late-interaction vectors — follows from it.

### What you will build

By the end of this notebook you will have a **working multilingual visual-document search engine**: you
generate a corpus of document page images, index them, query them in English / French / Spanish, measure
retrieval quality with real IR metrics, compress the index by ~100×, fine-tune the retriever on your own
labelled pairs, and bolt a VLM on top for visual RAG.

### Table of contents

| # | Section | What you learn |
|---|---------|----------------|
| 1 | [Why single-tower?](#s1) | The architecture, and what it replaces |
| 2 | [Setup](#s2) | Install, GPU check, model IDs |
| 3 | [The backbone](#s3) | `NeoMMEModel`: tokens in, hidden states out |
| 4 | [Masked language modelling](#s4) | `NeoMMEForMaskedLM`, multilingual fill-mask |
| 5 | [The image pipeline](#s5) | Patchification, resolution budgets, position IDs |
| 6 | [Sample data](#s6) | Generate a synthetic multilingual document corpus |
| 7 | [Retrieval](#s7) | `NeoMMEForRetrieval`, dense vs. late interaction |
| 8 | [A real search engine](#s8) | Index, query, rank, and *measure* (nDCG/MRR/Recall) |
| 9 | [Compression](#s9) | Matryoshka dims, token pooling, int8/binary quantization |
| 10 | [Fine-tuning](#s10) | Contrastive training in raw PyTorch and Sentence Transformers |
| 11 | [Visual RAG](#s11) | Retrieved page → VLM → grounded answer |
| 12 | [Production notes](#s12) | Throughput, batching, vector DBs, gotchas |
| 13 | [Troubleshooting & FAQ](#s13) | Every error you are likely to hit |

### How to run this

Runtime → Change runtime type → **T4 GPU** (or any GPU). CPU works for the 260M model but the
benchmark sections will be slow. Total runtime on a T4 is roughly **15–25 minutes**, most of it the
first model download.

Every section is self-contained after Section 2 (Setup) and Section 6 (Sample data) have run.

<a name="s1"></a>
## 1. Why a single tower? The problem NeoMME is solving

### 1.1 How multimodal retrievers are usually built

Almost every multimodal embedding model in production today is one of two shapes:

**Shape A — the dual tower (CLIP-style).** A separately pretrained image encoder (ViT) and a separately
pretrained text encoder, glued together by a contrastive loss. Two towers, two pretraining runs, two sets
of assumptions, and the image side never really learns to *read*.

**Shape B — the VLM adapter (ColPali-style).** Take a pretrained ViT, take a pretrained causal LLM, bolt
the ViT onto the LLM through a projector, then fine-tune the whole thing for retrieval. This reads
documents very well — it inherits the LLM's language ability — but you are now paying for a 3B–7B
*decoder* to do an *encoding* job, and the causal attention mask means every token only sees its past.

Both shapes inherit constraints from components that were never designed for the task:

* A ViT trained on 224×224 natural photographs is asked to read 2000×2000 pixel invoices.
* A causal decoder, whose entire architecture exists to predict the *next* token, is asked to produce
  a *bidirectional* summary of a page.
* Patch budgets are fixed by the vision tower's pretraining resolution, so high-resolution pages get
  tiled, downsampled, or both.

### 1.2 What NeoMME does instead

```
        ┌──────────────── Dual tower ────────────────┐   ┌────────── VLM adapter ──────────┐   ┌─── NeoMME ───┐
        │                                            │   │                                 │   │              │
text ──▶│ pretrained text encoder ─┐                 │   │            ┌── pretrained ──┐   │   │              │
        │                          ├─▶ cosine        │   │ text ─────▶│  causal  LLM   │   │   │  text ──┐    │
image ─▶│ pretrained ViT ──────────┘                 │   │ image ─▶ViT│  (causal mask) │   │   │         ├──▶ │  ONE
        │                                            │   │    └─proj─▶└────────────────┘   │   │ patches ┘    │  bidirectional
        └────────────────────────────────────────────┘   └─────────────────────────────────┘   │              │  Transformer
                                                                                                └──────────────┘
```

There is exactly one stack. Text enters through an **ALBERT-style factorized embedding**
(`vocab_size → embedding_rank=256 → hidden_size`), which keeps a 131k-token multilingual vocabulary cheap.
Images are cut into **non-overlapping 32×32 pixel patches**, each patch flattened to `3 × 32 × 32 = 3072`
numbers and pushed through a **2-layer MLP trained from scratch**. Both streams land in the same
`hidden_size` space and then attend to each other, bidirectionally, in every layer.

Because there is no vision tower, there is no vision-tower resolution to respect. The image is patchified
at its *native* resolution (optionally capped), and the model just gets a longer sequence. With a
**16,384-token context**, that is enough for roughly two 4K UHD pages in a single forward pass.

### 1.3 The training objective

NeoMME is not trained with next-token prediction. It is trained with a **masked discrete-diffusion**
objective: corrupt a fraction of the input (text tokens *and* image content), predict what was removed.
This is BERT's idea, generalized — the mask ratio varies rather than being pinned at 15%, so the model
learns to reconstruct from a whole spectrum of corruption levels. The result is an encoder whose
representations are bidirectional by construction, which is exactly what you want for retrieval.

### 1.4 The two heads that matter

`NeoMMEForRetrieval` is the backbone plus **two heads that run in the same forward pass**:

| Head | Output | Shape | Scored with |
|------|--------|-------|-------------|
| Multi-vector (late interaction) | `outputs.embeddings` | `(batch, seq_len, 128)` | `MeanMaxSim` |
| Dense (mean-pooled) | `outputs.dense_embeddings` | `(batch, hidden_size)` | Cosine similarity |

You do **not** choose at load time. You get both, from one pass over the page, and you can use the dense
vector for a cheap first-stage sweep over millions of pages and the multi-vector representation to rerank
the top few hundred. Section 8 builds exactly that pipeline.

### 1.5 The numbers

| | NeoMME-260M | NeoMME-800M |
|---|---|---|
| Parameters | ~260M | ~800M |
| Hidden size | 1024 | larger |
| Layers | 17 | more |
| Context | 16,384 tokens | 16,384 tokens |
| Patch size | 32×32 px | 32×32 px |
| Late-interaction dim | 128 | 128 |
| ViDoRe v3 nDCG@10 (Retriever) | **0.523** | **0.556** |
| Throughput @ 2048² on an L40S | ~51 pages/s | — |

The 260M figure is the interesting one: it beats **every** evaluated model strictly below 800M parameters
on ViDoRe v3, at roughly twice ColModernVBERT's throughput. And Section 9 reproduces the compression
result — hierarchical token pooling plus asymmetric quantization shrink a late-interaction index by up to
**255×** while keeping >95% of nDCG@10.

### 1.6 Attention: not all layers are equal

One more architectural detail you will see when you inspect the config. NeoMME interleaves two layer types:

* `sliding_attention` — local windows (alternating 256 and 1024 tokens), cheap, most of the stack.
* `full_attention` — every 6th layer and the final layer, global, the expensive-but-necessary glue.

Each type gets its *own* RoPE spectrum: sliding layers rotate every head dimension at short range
(θ = 10,000), global layers rotate only 25% of each head at long range (θ = 1,000,000) and leave the rest
unrotated for pure content matching. That is a deliberate division of labour between position and content,
and it is why a 16k context stays affordable.

<a name="s2"></a>
## 2. Setup

### 2.1 Install

NeoMME landed in Hugging Face Transformers on **2026-08-31**. Depending on when you run this, it may or may
not be in the latest PyPI release yet, so we install `transformers` from `main` — that always works. If your
installed release already has it, the git install is simply a no-op upgrade.

`sentence-transformers>=6.0.0` is needed for `mean_maxsim` (the MeanMaxSim scoring function) and for the
`MultiVectorEncoder` fine-tuning API used in Section 10.

> **Colab note:** after this cell finishes you may see a "restart session" prompt. You can ignore it —
> nothing here was imported before the install.

In [ ]:
%pip install -q -U accelerate "transformers @ git+https://github.com/huggingface/transformers.git@main" "sentence-transformers>=6.0.0" pillow matplotlib

print("✅ install complete — if Colab offers to restart the session, you can ignore it")

### 2.2 Environment check

Let's confirm we have what we need before downloading half a gigabyte of weights.

In [ ]:
import platform, sys

import torch
import transformers

print(f"python              {platform.python_version()}")
print(f"torch               {torch.__version__}")
print(f"transformers        {transformers.__version__}")

try:
    import sentence_transformers
    print(f"sentence-transformers {sentence_transformers.__version__}")
except ImportError:
    print("sentence-transformers NOT installed (sections 9-10 need it)")

# Does this transformers build actually know about NeoMME?
try:
    from transformers import NeoMMEConfig, NeoMMEForMaskedLM, NeoMMEForRetrieval, NeoMMEModel, NeoMMEProcessor
    print("\n✅ NeoMME classes are available")
except ImportError as exc:  # pragma: no cover - environment dependent
    raise SystemExit(
        "❌ This transformers build has no NeoMME support.\n"
        "   Re-run the install cell, then restart the runtime.\n"
        f"   ({exc})"
    )

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"\n🖥️  GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    DEVICE = "cpu"
    print("\n⚠️  No GPU detected — everything still runs, benchmarks will just be slow.")

# bf16 on Ampere+, fp16 on older GPUs, fp32 on CPU.
if DEVICE == "cuda":
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DTYPE = torch.float32
print(f"    dtype: {DTYPE}")

### 2.3 The model IDs

All checkpoints are **Apache 2.0** and live in the
[NeoMME collection](https://huggingface.co/collections/Hcompany/neomme).

| Checkpoint | Class | Use it for |
|---|---|---|
| `Hcompany/NeoMME-260M` | `NeoMMEModel` / `NeoMMEForMaskedLM` | Backbone: hidden states, fill-mask, your own fine-tuning |
| `Hcompany/NeoMME-800M` | `NeoMMEModel` / `NeoMMEForMaskedLM` | Same, bigger |
| `Hcompany/NeoMME-260M-Retriever` | `NeoMMEForRetrieval` | Visual document retrieval (dense + late interaction) |
| `Hcompany/NeoMME-800M-Retriever` | `NeoMMEForRetrieval` | Same, bigger, +0.033 nDCG@10 on ViDoRe v3 |
| `Hcompany/NeoMME-{260M,800M}-Pretrain-predecay-s450000` | `NeoMMEModel` | Pre-LR-decay checkpoints, for researchers continuing pretraining |

We default to the 260M pair so this notebook runs comfortably on a free T4. Flip `MODEL_SIZE` to `"800M"`
if you have the VRAM — nothing else in the notebook changes.

> The 260M IDs above are the ones used in the official Transformers documentation. If an 800M ID 404s,
> check the collection for its exact name and set `BACKBONE_ID` / `RETRIEVER_ID` directly.

In [ ]:
MODEL_SIZE = "260M"          # "260M" or "800M"

BACKBONE_ID  = f"Hcompany/NeoMME-{MODEL_SIZE}"
RETRIEVER_ID = f"Hcompany/NeoMME-{MODEL_SIZE}-Retriever"

print(f"backbone : {BACKBONE_ID}")
print(f"retriever: {RETRIEVER_ID}")

### 2.4 Shared imports and helpers

A few small utilities we reuse throughout. Nothing clever — just enough to keep the interesting cells short.

In [ ]:
import time
import textwrap
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

torch.manual_seed(0)
np.random.seed(0)


def show_images(images, titles=None, cols=4, width=3.2, title_size=9):
    """Grid-display a list of PIL images inline."""
    images = list(images)
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(width * cols, width * 1.35 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, image in zip(axes, images):
        ax.imshow(image)
    if titles is not None:
        for ax, title in zip(axes, titles):
            ax.set_title(textwrap.fill(str(title), 34), fontsize=title_size)
    plt.tight_layout()
    plt.show()


def human_bytes(n):
    """1234567 -> '1.2 MB'."""
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024


def param_count(model):
    return sum(p.numel() for p in model.parameters())


class Timer:
    """with Timer("encode") as t: ...   ->  t.seconds"""

    def __init__(self, label=""):
        self.label = label

    def __enter__(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.seconds = time.perf_counter() - self.start
        if self.label:
            print(f"⏱️  {self.label}: {self.seconds:.3f}s")


print("✅ helpers ready")

<a name="s3"></a>
## 3. The backbone: `NeoMMEModel`

We start at the bottom of the stack. `NeoMMEModel` is the pure encoder: tokens and patches go in,
contextual hidden states come out. Everything else in this notebook is this model plus a head.

### 3.1 Load the model and the processor

`NeoMMEProcessor` bundles two things: a **tokenizer** (a 131k-entry multilingual vocabulary with a handful
of special markers) and a **`NeoMMEImageProcessor`** (which turns pixels into a flat table of patches).
You almost never touch them separately.

In [ ]:
from transformers import AutoModel, AutoProcessor

processor = AutoProcessor.from_pretrained(BACKBONE_ID)
model = AutoModel.from_pretrained(BACKBONE_ID, dtype=DTYPE).to(DEVICE).eval()

print(f"model class     : {type(model).__name__}")
print(f"processor class : {type(processor).__name__}")
print(f"tokenizer class : {type(processor.tokenizer).__name__}")
print(f"image processor : {type(processor.image_processor).__name__}")
print(f"parameters      : {param_count(model):,}")

### 3.2 Read the config — the architecture in numbers

Everything claimed in Section 1 is verifiable from `model.config`. Let's actually check it rather than
take it on faith.

In [ ]:
cfg = model.config

print("── core dimensions ──────────────────────────────")
print(f"hidden_size              {cfg.hidden_size}")
print(f"num_hidden_layers        {cfg.num_hidden_layers}")
print(f"num_attention_heads      {cfg.num_attention_heads}")
print(f"num_key_value_heads      {cfg.num_key_value_heads}   (grouped-query attention)")
print(f"head_dim                 {cfg.head_dim}")
print(f"intermediate_size        {cfg.intermediate_size}")
print(f"hidden_act               {cfg.hidden_act}")

print("\n── the factorized text embedding ────────────────")
print(f"vocab_size               {cfg.vocab_size:,}")
print(f"embedding_rank           {cfg.embedding_rank}   (ALBERT-style bottleneck)")
naive = cfg.vocab_size * cfg.hidden_size
factorized = cfg.vocab_size * cfg.embedding_rank + cfg.embedding_rank * cfg.hidden_size
print(f"  naive  vocab x hidden  {naive:,} params")
print(f"  actual vocab x rank    {factorized:,} params  ({naive / factorized:.1f}x smaller)")

print("\n── the image path (no vision tower!) ────────────")
print(f"patch_size               {cfg.patch_size} px")
print(f"patch_dim                {cfg.patch_dim}   (= 3 * {cfg.patch_size}^2, one flattened RGB patch)")

print("\n── context and positions ────────────────────────")
print(f"max_position_embeddings  {cfg.max_position_embeddings:,} tokens")
# `sliding_window` is per-layer, so reading cfg.sliding_window directly raises.
windows = sorted({cfg.per_layer_config[i].sliding_window for i in range(cfg.num_hidden_layers)},
                 key=lambda w: (w is None, w))
print(f"sliding windows          {windows}   (None = global attention)")
print(f"embedding_dim            {cfg.embedding_dim}   (late-interaction vector width)")

### 3.3 The attention schedule

`layer_types` is where the sliding/global interleaving lives. Note that **the last layer is always
global** — the final representation needs every token to have seen every other token.

In [ ]:
schedule = cfg.layer_types
glyphs = {"sliding_attention": "▪", "full_attention": "█"}

print("layer:  " + "".join(f"{i:>3}" for i in range(len(schedule))))
print("type :  " + "".join(f"  {glyphs[t]}" for t in schedule))
print("\n█ = full (global) attention    ▪ = sliding-window attention\n")

for layer_type in sorted(set(schedule)):
    idx = [i for i, t in enumerate(schedule) if t == layer_type]
    rope = cfg.rope_parameters[layer_type]
    print(f"{layer_type:20s} layers {idx}")
    print(f"{'':20s} rope_theta={rope['rope_theta']:>12,.0f}  "
          f"partial_rotary_factor={rope['partial_rotary_factor']}  "
          f"-> rotates {int(cfg.head_dim * rope['partial_rotary_factor'])}/{cfg.head_dim} head dims")

# Per-layer sliding windows alternate short/long.
print("\nper-layer sliding windows:")
for i, layer_type in enumerate(schedule):
    window = cfg.per_layer_config[i].sliding_window
    print(f"  layer {i:>2}  {layer_type:20s} window={window}")

Read that output carefully — it is the whole efficiency story in one place:

* **Sliding layers** (`θ = 10,000`, `partial_rotary_factor = 1.0`) rotate *every* head dimension. Position
  is everything at short range; a patch two rows up is a different thing from a patch two columns left.
* **Global layers** (`θ = 1,000,000`, `partial_rotary_factor = 0.25`) rotate only a quarter of each head
  and leave the remaining 48 dimensions **unrotated**. Those unrotated dimensions do pure *content*
  matching across the whole 16k context — "is there a number that looks like this anywhere on the page?"
* Sliding windows **alternate 256 / 1024**, so information propagates at two different speeds.

### 3.4 The special tokens

NeoMME's vocabulary reserves the first few IDs for structural markers. These are not decoration — the
model's forward pass *validates* their layout and will raise if you get it wrong.

In [ ]:
tok = processor.tokenizer

markers = {
    "document_token": "prefix for anything indexed as a document (`<doc>`)",
    "image_token": "one per image patch, plus one leading marker (`<img>`)",
    "row_token": "end-of-row marker in the patch grid (`<row>`)",
    "query_token": "prefix for search queries (`<query>`)",
    "mask_token": "the masked-LM / query-expansion slot (`<mask>`)",
    "pad_token": "padding",
}

print(f"{'attribute':<16} {'token':<10} {'id':>6}   role")
print("-" * 92)
for name, role in markers.items():
    token = getattr(tok, name, None)
    token_id = getattr(tok, f"{name}_id", None)
    if token is None:
        print(f"{name:<16} {'(absent)':<10} {'-':>6}   {role}")
    else:
        print(f"{name:<16} {token:<10} {token_id:>6}   {role}")

print(f"\nconfig cross-check: document_token_id={cfg.document_token_id}, "
      f"image_token_id={cfg.image_token_id}, pad_token_id={cfg.pad_token_id}")

### 3.5 Encoding text

`processor(text=...)` tokenizes **exactly what you give it** — it adds no markers of its own. That is a
deliberate design choice: the marker you need depends on the task.

* Masked LM → prepend `tokenizer.document_token` yourself.
* Retrieval → use `apply_chat_template(..., task="query" | "document")` (Section 7).

Here is the helper the official docs use for the first case.

In [ ]:
def encode_document_text(processor, text: str) -> str:
    """Prefix plain text with the <doc> marker, as the backbone expects."""
    return f"{processor.tokenizer.document_token}{text}"


sentences = [
    "The cat sat on a mat.",
    "Le chat était assis sur un tapis.",          # French
    "El gato se sentó en una alfombra.",          # Spanish
    "Die Katze saß auf einer Matte.",             # German
    "猫はマットの上に座っていた。",                    # Japanese
]

inputs = processor(
    text=[encode_document_text(processor, s) for s in sentences],
    padding=True,
    return_tensors="pt",
).to(DEVICE)

print("processor returned:", {k: tuple(v.shape) for k, v in inputs.items()})

with torch.inference_mode():
    outputs = model(**inputs)

print(f"\nlast_hidden_state: {tuple(outputs.last_hidden_state.shape)}  "
      f"= (batch, seq_len, hidden_size)")
print("\ntoken-by-token view of the first sentence:")
ids = inputs.input_ids[0].tolist()
mask = inputs.attention_mask[0].tolist()
for position, (token_id, is_real) in enumerate(zip(ids, mask)):
    if not is_real:
        continue
    piece = tok.convert_ids_to_tokens(token_id)
    print(f"  [{position:>2}] id={token_id:<7} {piece!r}")

Notice that the hidden states are **bidirectional**: token 0's representation already depends on token 12.
There is no causal mask anywhere in this model. That is why a single mean-pool over these states is a
respectable sentence embedding, and why the retrieval head in Section 7 can get away with being so simple.

### 3.6 Encoding an image

Now the interesting half. We hand the processor a picture and *no text at all*. It builds the document
prompt for us: `<doc>` `<img>` followed by the patch grid, with `<row>` markers separating rows.

In [ ]:
def make_demo_page(width=384, height=256):
    """A tiny synthetic 'document page' so this cell needs no network."""
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    draw.rectangle([0, 0, width - 1, 47], fill=(30, 64, 120))
    draw.text((14, 16), "QUARTERLY REPORT", fill="white")
    draw.text((14, 70), "Revenue grew 18% year over year,", fill=(20, 20, 20))
    draw.text((14, 90), "driven by the EMEA region.", fill=(20, 20, 20))
    for i, h in enumerate([40, 65, 55, 90]):
        draw.rectangle([30 + i * 60, 220 - h, 70 + i * 60, 220], fill=(200, 80, 60))
    draw.line([20, 220, width - 20, 220], fill="black", width=2)
    return image


page = make_demo_page()
display(page)

image_inputs = processor(images=[page], return_tensors="pt").to(DEVICE)

print("\nprocessor returned:")
for key, value in image_inputs.items():
    print(f"  {key:<15} {tuple(value.shape)}  {value.dtype}")

In [ ]:
# What did the processor actually put in input_ids?
ids = image_inputs.input_ids[0].tolist()
id_to_name = {
    tok.document_token_id: "<doc>",
    tok.image_token_id: "<img>",
    tok.row_token_id: "<row>",
}

grid_h = page.height // cfg.patch_size + (page.height % cfg.patch_size > 0)
grid_w = page.width // cfg.patch_size + (page.width % cfg.patch_size > 0)

print(f"image {page.width}x{page.height} px, patch_size={cfg.patch_size}")
print(f"  -> patch grid {grid_h} rows x {grid_w} cols = {grid_h * grid_w} patches")
print(f"  -> sequence   1 <doc> + 1 <img> + {grid_h}*({grid_w} <img> + 1 <row>) "
      f"= {2 + grid_h * (grid_w + 1)} tokens")
print(f"  -> actual     {len(ids)} tokens  "
      f"{'✅' if len(ids) == 2 + grid_h * (grid_w + 1) else '❌'}")

print("\nfirst 3 rows of the token layout:")
head = ids[: 2 + 3 * (grid_w + 1)]
print("  " + " ".join(id_to_name.get(i, "?") for i in head[:2]))
offset = 2
for row in range(3):
    row_ids = ids[offset : offset + grid_w + 1]
    print(f"  row {row}: " + " ".join(id_to_name.get(i, "?") for i in row_ids))
    offset += grid_w + 1

print(f"\npixel_values {tuple(image_inputs.pixel_values.shape)} "
      f"= (total_patches, 3 * patch_size^2)")
print(f"position_ids {tuple(image_inputs.position_ids.shape)} = (axes, batch, seq_len)")

### 3.7 Two-axis positions (M-RoPE)

That `position_ids` tensor has a leading dimension of **2**. This is the multimodal RoPE: instead of one
scalar position per token, each token gets a **(row, column)** pair.

* Text tokens get `(i, i)` — the two axes agree, so it degenerates to ordinary 1-D RoPE.
* Image patches get their true `(row, col)` in the grid, offset by 2 for the `<doc><img>` prefix.

This is why the model knows that the patch directly *above* another patch is spatially adjacent, even
though they are hundreds of tokens apart in the flattened sequence.

In [ ]:
pos = image_inputs.position_ids[:, 0].cpu()          # (2, seq_len)
print("token index :", list(range(10)))
print("axis 0 (row):", pos[0, :10].tolist())
print("axis 1 (col):", pos[1, :10].tolist())

print(f"\n<doc> at index 0 -> ({pos[0,0]}, {pos[1,0]})   <img> at index 1 -> ({pos[0,1]}, {pos[1,1]})")
print(f"first grid patch  -> ({pos[0,2]}, {pos[1,2]})   (row 0, col 0, both offset by 2)")

# Visualise the row axis: you should see clean horizontal bands, one per patch row.
row_axis = pos[0, 2:].numpy()
col_axis = pos[1, 2:].numpy()
fig, axes = plt.subplots(1, 2, figsize=(13, 2.6))
for ax, values, name in zip(axes, [row_axis, col_axis], ["axis 0 (row)", "axis 1 (column)"]):
    ax.plot(values, lw=0.9)
    ax.set_title(f"{name} across the flattened patch sequence")
    ax.set_xlabel("position in sequence")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The left plot is a **staircase** (one step per patch row) and the right is a **sawtooth** (resetting at
each row boundary). That is the 2-D grid, unrolled.

### 3.8 Text and images in one batch

The docs' opening example does something worth pausing on: it puts a *sentence* and a *page image* in the
same batch, in the same forward pass, through the same weights. No modality routing, no separate encoder.

In [ ]:
mixed_inputs = processor(
    text=[
        encode_document_text(processor, "The cat sat on a mat."),
        encode_document_text(processor, processor.image_token),   # placeholder, expanded to the patch grid
    ],
    images=[page],
    padding=True,
    return_tensors="pt",
).to(DEVICE)

with torch.inference_mode():
    mixed_outputs = model(**mixed_inputs)

text_hidden_states, image_hidden_states = mixed_outputs.last_hidden_state

print(f"batch last_hidden_state : {tuple(mixed_outputs.last_hidden_state.shape)}")
print(f"  text row              : {tuple(text_hidden_states.shape)}")
print(f"  image row             : {tuple(image_hidden_states.shape)}")
print("\nSame weights, same forward pass, two modalities. That is the whole point of a single tower.")

### 3.9 Where the patches actually enter

Worth seeing once: the patch stem is a **2-layer MLP**, 4 tensors, ~7M parameters. That is the entire
"vision encoder".

In [ ]:
print("patch stem (NeoMMEPatchEmbeddings):")
stem_params = 0
for name, tensor in model.patch_embeddings.named_parameters():
    print(f"  {name:<20} {tuple(tensor.shape)}")
    stem_params += tensor.numel()
print(f"\n  total: {stem_params:,} parameters "
      f"({100 * stem_params / param_count(model):.1f}% of the model)")

# Project patches by hand, exactly as forward() does internally.
with torch.inference_mode():
    features = model.get_image_features(image_inputs.pixel_values.to(DTYPE))
print(f"\nget_image_features: {tuple(image_inputs.pixel_values.shape)} "
      f"-> {tuple(features.last_hidden_state.shape)}")
print("(num_patches, 3072 raw pixels) -> (num_patches, hidden_size)")

<a name="s4"></a>
## 4. Masked language modelling: the pretraining objective, exposed

`NeoMMEForMaskedLM` puts the pretraining head back on the backbone. This is worth doing even if you only
care about retrieval, for two reasons:

1. It is the **cheapest possible sanity check** that your weights loaded correctly and your prompt
   formatting is right. If fill-mask returns garbage, nothing downstream will work.
2. It is how you'd do **domain-adaptive continued pretraining** on your own unlabelled corpus before
   fine-tuning a retriever (Section 10).

### 4.1 Fill in the blank

In [ ]:
from transformers import AutoModelForMaskedLM

mlm = AutoModelForMaskedLM.from_pretrained(BACKBONE_ID, dtype=DTYPE).to(DEVICE).eval()
print(f"loaded {type(mlm).__name__}  ({param_count(mlm):,} params)")

# The MLM head is *tied* to the factorized embedding — it costs almost nothing.
print("\ntied weights:", mlm._tied_weights_keys)

In [ ]:
def fill_mask(text_with_mask: str, top_k: int = 5):
    """Predict the top-k tokens for every <mask> in `text_with_mask`."""
    text = f"{tok.document_token}{text_with_mask}"
    inputs = processor(text=[text], return_tensors="pt").to(DEVICE)

    with torch.inference_mode():
        logits = mlm(**inputs).logits

    positions = (inputs.input_ids[0] == tok.mask_token_id).nonzero().flatten().tolist()
    if not positions:
        raise ValueError(f"no {tok.mask_token} found in the input")

    results = []
    for position in positions:
        probs = torch.softmax(logits[0, position].float(), dim=-1)
        scores, ids = probs.topk(top_k)
        results.append([(tok.decode(i), float(s)) for i, s in zip(ids, scores)])
    return results


# The doc example, verbatim: "<doc>The capital of <mask> is London."
predictions = fill_mask(f"The capital of {tok.mask_token} is London.")
print("The capital of ___ is London.")
for token, score in predictions[0]:
    bar = "█" * int(score * 40)
    print(f"  {token!r:<18} {score:6.2%}  {bar}")

### 4.2 It really is multilingual

NeoMME's 131k vocabulary was trained on multilingual data from scratch. Let's confirm that by masking the
same *kind* of fact in several languages — no translation layer, no language ID, just one model.

In [ ]:
probes = [
    ("English",  f"Paris is the capital of {tok.mask_token}."),
    ("French",   f"Paris est la capitale de la {tok.mask_token}."),
    ("Spanish",  f"París es la capital de {tok.mask_token}."),
    ("German",   f"Paris ist die Hauptstadt von {tok.mask_token}."),
    ("Italian",  f"Parigi è la capitale della {tok.mask_token}."),
    ("Portuguese", f"Paris é a capital da {tok.mask_token}."),
]

for language, probe in probes:
    top = fill_mask(probe, top_k=3)[0]
    rendered = ", ".join(f"{t!r} ({s:.0%})" for t, s in top)
    print(f"{language:<12} {probe.replace(tok.mask_token, '___')}")
    print(f"{'':<12} -> {rendered}\n")

### 4.3 Multiple masks, and why bidirectionality matters

A causal LM fills blanks left to right and cannot use anything to the right of the mask. NeoMME sees the
whole sentence. Watch what happens when the *only* disambiguating evidence comes **after** the mask.

In [ ]:
sentences = [
    f"The {tok.mask_token} barked loudly at the postman.",         # evidence after the mask
    f"She opened her {tok.mask_token} and started to read.",
    f"The {tok.mask_token} was too salty, so I sent it back.",
    f"He put the letter in an {tok.mask_token} and posted it.",
]

for sentence in sentences:
    top = fill_mask(sentence, top_k=3)[0]
    print(sentence.replace(tok.mask_token, "___"))
    print("   -> " + ", ".join(f"{t!r} ({s:.0%})" for t, s in top) + "\n")

# Several masks at once — the model resolves them jointly, not sequentially.
multi = f"The {tok.mask_token} of France is {tok.mask_token}."
print(multi.replace(tok.mask_token, "___"))
for i, slot in enumerate(fill_mask(multi, top_k=3)):
    print(f"   mask {i}: " + ", ".join(f"{t!r} ({s:.0%})" for t, s in slot))

### 4.4 Scoring text with pseudo-log-likelihood

A neat trick the MLM head gives you for free: mask each token in turn and read off its probability. Sum the
logs and you have a **pseudo-log-likelihood** — a usable fluency / plausibility score, and a good way to
probe what the model actually believes.

In [ ]:
def pseudo_log_likelihood(text: str) -> float:
    """Average log P(token | rest of sentence), one masked position at a time."""
    inputs = processor(text=[f"{tok.document_token}{text}"], return_tensors="pt")
    ids = inputs.input_ids[0]
    # Skip the <doc> marker at index 0.
    positions = list(range(1, len(ids)))

    batch = ids.unsqueeze(0).repeat(len(positions), 1).clone()
    for row, position in enumerate(positions):
        batch[row, position] = tok.mask_token_id
    batch = batch.to(DEVICE)

    with torch.inference_mode():
        logits = mlm(input_ids=batch,
                     attention_mask=torch.ones_like(batch)).logits.float()

    total = 0.0
    for row, position in enumerate(positions):
        log_probs = torch.log_softmax(logits[row, position], dim=-1)
        total += float(log_probs[ids[position]])
    return total / len(positions)


candidates = [
    "The invoice total is due within thirty days.",
    "The invoice total is due within thirty bananas.",
    "Revenue increased by 18% compared to last year.",
    "Revenue increased by 18% compared to last banana.",
]
for candidate in candidates:
    print(f"{pseudo_log_likelihood(candidate):8.3f}   {candidate}")
print("\n(higher = more plausible under the model)")

### 4.5 Housekeeping

The MLM head is only needed for this section. Free it before we load the retriever.

In [ ]:
del mlm
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("✅ MLM head released")

<a name="s5"></a>
## 5. The image pipeline in depth

Because NeoMME has no vision tower, the image processor is doing something much simpler — and much more
transparent — than a ViT preprocessor. It is worth understanding precisely, because **it is the single
biggest lever you have over cost and quality**.

The pipeline is:

```
PIL image
   │  1. convert to RGB
   │  2. optionally resize   (max_side, and/or a min_pixels..max_pixels area budget)
   │  3. pad bottom + right  to a whole multiple of patch_size  (padding becomes exact black)
   │  4. rescale to [0,1] and normalize with ImageNet mean/std
   │  5. cut into row-major 32x32 patches, flatten each to 3*32*32 = 3072 floats
   ▼
pixel_values  (total_patches, 3072)    +    image_grid_hw  (num_images, 2)
```

Note the ordering of 3 and 4: **padding happens before rescaling**, so padded pixels normalize to exactly
the same value as a black canvas. That is deliberate — it matches the reference implementation.

### 5.1 Patchification, made visible

Let's prove step 5 by *inverting* it: take `pixel_values` straight out of the processor, un-normalize, and
reassemble the image. If we get the original page back, we understand the format completely.

In [ ]:
from transformers.utils.constants import IMAGENET_STANDARD_MEAN, IMAGENET_STANDARD_STD


def patches_to_image(pixel_values, grid_hw, patch_size):
    """Invert NeoMMEImageProcessor: (num_patches, 3*p*p) -> PIL image."""
    grid_h, grid_w = int(grid_hw[0]), int(grid_hw[1])
    patches = pixel_values[: grid_h * grid_w].float().cpu().numpy()
    # (patches, p, p, 3) — the processor flattens as (h, w, c) within a patch.
    patches = patches.reshape(grid_h, grid_w, patch_size, patch_size, 3)
    # Stitch rows and columns back together.
    canvas = patches.transpose(0, 2, 1, 3, 4).reshape(grid_h * patch_size, grid_w * patch_size, 3)
    canvas = canvas * np.array(IMAGENET_STANDARD_STD) + np.array(IMAGENET_STANDARD_MEAN)
    return Image.fromarray((np.clip(canvas, 0, 1) * 255).astype(np.uint8))


single = processor.image_processor([page], return_tensors="pt")
grid = single.image_grid_hw[0]
reconstructed = patches_to_image(single.pixel_values, grid, cfg.patch_size)

print(f"pixel_values  {tuple(single.pixel_values.shape)}")
print(f"image_grid_hw {single.image_grid_hw.tolist()}  (rows, cols)")
print(f"original {page.size}  ->  reconstructed {reconstructed.size} "
      f"(padded up to a whole patch grid)")

show_images([page, reconstructed],
            ["original", "reconstructed from pixel_values"], cols=2, width=4.5)

The reconstruction is pixel-identical except for the padding strip on the bottom/right edge. There is no
information loss in patchification — the model sees the raw pixels, arranged in a grid.

### 5.2 Sequence length is quadratic in resolution — budget accordingly

The one thing that will bite you in production: **doubling image resolution quadruples the token count**.
`get_number_of_image_patches` tells you the cost before you pay it.

In [ ]:
image_processor = processor.image_processor
patch = cfg.patch_size

print(f"{'page size':<16}{'patches':>10}{'+row toks':>11}{'total':>9}   {'% of 16k ctx':>13}")
print("-" * 64)
common_sizes = [
    ("thumbnail 512²",   512,  512),
    ("web 1024²",       1024, 1024),
    ("A4 @ 150 dpi",    1240, 1754),
    ("A4 @ 200 dpi",    1654, 2339),
    ("benchmark 2048²", 2048, 2048),
    ("4K UHD",          3840, 2160),
]
for label, width, height in common_sizes:
    grid_h, grid_w = -(-height // patch), -(-width // patch)
    patches = image_processor.get_number_of_image_patches(height, width)
    total = 2 + grid_h * (grid_w + 1)
    share = total / cfg.max_position_embeddings
    flag = "  ⚠️ over context" if total > cfg.max_position_embeddings else ""
    print(f"{label:<16}{patches:>10,}{grid_h:>11,}{total:>9,}   {share:>12.1%}{flag}")

print(f"\nContext window: {cfg.max_position_embeddings:,} tokens "
      f"-> about two 4K pages, exactly as advertised.")

### 5.3 The three resize knobs

`NeoMMEImageProcessor` accepts three ways to cap cost. They compose — the smallest resulting scale wins.

| Knob | Meaning | When to reach for it |
|---|---|---|
| `max_side` | longest side, in pixels | Simple, predictable. The usual default. |
| `size={"max_pixels": N}` | total area, in pixels | Mixed aspect ratios — a wide chart and a tall page get the same budget. |
| `size={"min_pixels": N}` | *upscale* small images to at least N pixels | Thumbnails / low-res scans that would otherwise become 4 patches. |
| `patch_size` | patch side | Leave it alone — it must match `config.patch_size`. |

By default the checkpoint's own settings apply and native resolution is preserved.

In [ ]:
print(f"checkpoint defaults: max_side={image_processor.max_side}, size={image_processor.size}")

big = make_demo_page(1600, 1200)
configs = [
    ("native (no cap)",             {}),
    ("max_side=768",                {"max_side": 768}),
    ("max_side=512",                {"max_side": 512}),
    ("area cap 512x512 px",         {"size": {"min_pixels": 1, "max_pixels": 512 * 512}}),
    ("area cap + max_side=640",     {"size": {"min_pixels": 1, "max_pixels": 512 * 512}, "max_side": 640}),
]

print(f"\n{'setting':<28}{'grid (hxw)':>13}{'patches':>10}{'seq len':>10}{'vs native':>11}")
print("-" * 74)
native_tokens = None
for label, kwargs in configs:
    out = processor.image_processor([big], return_tensors="pt", **kwargs)
    grid_h, grid_w = out.image_grid_hw[0].tolist()
    patches = grid_h * grid_w
    seq = 2 + grid_h * (grid_w + 1)
    native_tokens = native_tokens or seq
    print(f"{label:<28}{f'{grid_h}x{grid_w}':>13}{patches:>10,}{seq:>10,}{seq / native_tokens:>10.1%}")

> **Rule of thumb for document retrieval.** Text has to stay legible after resizing. A 10 pt character at
> 150 dpi is about 20 px tall, which is well inside one 32 px patch — good. Squash the same page to
> `max_side=512` and that character is ~7 px, spread across a fraction of a patch — the model can see that
> *something* is written there, but not what. If your queries depend on reading small print, keep
> `max_side` at 1024 or above and pay the tokens.

### 5.4 Batching images of different sizes

A subtlety the API enforces: when a batch contains images, every row must be padded to a common length
(`padding="longest"` or `padding=True`), because `position_ids` is built as a dense `(2, batch, seq_len)`
tensor. Meanwhile `pixel_values` is *concatenated*, not stacked — one flat table of patches for the whole
batch, indexed by `image_grid_hw`.

In [ ]:
pages = [make_demo_page(320, 200), make_demo_page(512, 384), make_demo_page(256, 256)]
batch = processor(images=pages, padding=True, return_tensors="pt")

print(f"input_ids     {tuple(batch.input_ids.shape)}   (padded to the longest row)")
print(f"attention_mask{tuple(batch.attention_mask.shape)}")
print(f"position_ids  {tuple(batch.position_ids.shape)}")
print(f"pixel_values  {tuple(batch.pixel_values.shape)}   <- CONCATENATED across images\n")

offset = 0
for i, image in enumerate(pages):
    grid_h, grid_w = batch.image_grid_hw[i].tolist() if "image_grid_hw" in batch else (None, None)
    real = int(batch.attention_mask[i].sum())
    print(f"  image {i}: {str(image.size):<12} real tokens={real:<5} padded to {batch.input_ids.shape[1]}")

print("\npixel_values rows are sliced per image using image_grid_hw — "
      "the model does that for you inside forward().")

<a name="s6"></a>
## 6. Sample data: a synthetic multilingual document corpus

Everything from here on needs a corpus. Rather than download a benchmark (network-dependent, licence-
encumbered, and impossible to inspect), we **render our own document pages** with PIL. This gives us:

* **Ground truth we control.** We know exactly which page answers which query, so we can compute real IR
  metrics in Section 8 instead of eyeballing results.
* **Genuine visual-document difficulty.** The answers live in bar charts, tables and body text — you have
  to *read the page image* to find them. No OCR, no metadata, no captions.
* **Multilinguality.** Pages in English, French, Spanish, German, Italian and Portuguese, so we can test
  cross-lingual retrieval (English query → French page).

### 6.1 A tiny document renderer

Four primitives — heading, paragraph block, bar chart, table — composed from a declarative page spec.

In [ ]:
FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/System/Library/Fonts/Supplemental/Arial.ttf",
    "C:\\Windows\\Fonts\\arial.ttf",
]


def load_font(size, bold=False):
    """Best-effort TrueType font, falling back to PIL's bitmap default."""
    candidates = FONT_CANDIDATES[1:2] + FONT_CANDIDATES if bold else FONT_CANDIDATES
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    try:
        return ImageFont.load_default(size=size)   # Pillow >= 10.1
    except TypeError:
        return ImageFont.load_default()


PAGE_W, PAGE_H = 900, 1200
INK = (24, 24, 28)
ACCENT = (28, 62, 120)
MUTED = (110, 112, 120)
PALETTE = [(214, 92, 62), (58, 124, 165), (94, 158, 106), (200, 160, 62), (140, 96, 168)]


def render_page(spec):
    """Render one document page from a declarative spec dict."""
    image = Image.new("RGB", (PAGE_W, PAGE_H), (252, 252, 250))
    draw = ImageDraw.Draw(image)
    f_title, f_sub = load_font(38, bold=True), load_font(20)
    f_h2, f_body, f_small = load_font(26, bold=True), load_font(21), load_font(17)

    # Header band
    draw.rectangle([0, 0, PAGE_W, 130], fill=ACCENT)
    draw.text((48, 34), spec["title"], font=f_title, fill="white")
    draw.text((48, 86), spec.get("subtitle", ""), font=f_sub, fill=(206, 218, 236))

    y = 170
    for block in spec["blocks"]:
        kind = block["kind"]

        if kind == "heading":
            draw.text((48, y), block["text"], font=f_h2, fill=ACCENT)
            y += 44

        elif kind == "text":
            for line in textwrap.wrap(block["text"], width=block.get("width", 62)):
                draw.text((48, y), line, font=f_body, fill=INK)
                y += 30
            y += 14

        elif kind == "bars":
            labels, values = block["labels"], block["values"]
            chart_h, base_y, left = 300, y + 300, 90
            slot = (PAGE_W - 150) / len(values)
            scale = chart_h / (max(values) * 1.15)
            for i, (label, value) in enumerate(zip(labels, values)):
                x0 = left + i * slot + slot * 0.18
                x1 = left + (i + 1) * slot - slot * 0.18
                draw.rectangle([x0, base_y - value * scale, x1, base_y],
                               fill=PALETTE[i % len(PALETTE)])
                draw.text((x0, base_y + 8), str(label), font=f_small, fill=MUTED)
                draw.text((x0, base_y - value * scale - 24), str(value), font=f_small, fill=INK)
            draw.line([left - 12, base_y, PAGE_W - 48, base_y], fill=INK, width=2)
            draw.line([left - 12, y, left - 12, base_y], fill=INK, width=2)
            draw.text((48, y - 34), block.get("caption", ""), font=f_small, fill=MUTED)
            y = base_y + 60

        elif kind == "table":
            header, rows = block["header"], block["rows"]
            col_w = (PAGE_W - 96) / len(header)
            draw.rectangle([48, y, PAGE_W - 48, y + 38], fill=(232, 236, 242))
            for c, name in enumerate(header):
                draw.text((58 + c * col_w, y + 9), str(name), font=f_small, fill=ACCENT)
            y += 38
            for r, row in enumerate(rows):
                if r % 2:
                    draw.rectangle([48, y, PAGE_W - 48, y + 34], fill=(245, 246, 248))
                for c, cell in enumerate(row):
                    draw.text((58 + c * col_w, y + 7), str(cell), font=f_small, fill=INK)
                y += 34
            draw.line([48, y, PAGE_W - 48, y], fill=(190, 192, 198), width=1)
            y += 30

    draw.text((48, PAGE_H - 46), spec.get("footer", ""), font=f_small, fill=MUTED)
    return image


print("✅ renderer ready")

### 6.2 The corpus

Twelve pages, six languages, three document types (charts, tables, prose). Each carries **one specific
fact** that a query will have to find.

In [ ]:
CORPUS_SPECS = [
    dict(
        id="energy_hourly", lang="en", title="National Grid Report 2019",
        subtitle="Hourly electricity generation profile",
        fact="Peak electricity generation in 2019 occurred at 18:00 with 47.2 GW.",
        blocks=[
            dict(kind="heading", text="Generation by hour of day (GW)"),
            dict(kind="bars", caption="Average output, 2019",
                 labels=["00h", "04h", "08h", "12h", "16h", "18h", "22h"],
                 values=[21.4, 19.8, 33.6, 41.0, 44.5, 47.2, 28.9]),
            dict(kind="text", text="Demand peaked in the early evening, when domestic heating "
                                   "and lighting load coincided with residual industrial demand."),
        ],
        footer="National Grid / Statistics Division",
    ),
    dict(
        id="kazakhstan_oil", lang="fr", title="Production pétrolière du Kazakhstan",
        subtitle="Rapport annuel sur les hydrocarbures",
        fact="22% of Kazakhstan's oil production comes from offshore fields.",
        blocks=[
            dict(kind="heading", text="Répartition de la production"),
            dict(kind="text", text="La production totale a atteint 90,5 millions de tonnes. "
                                   "Les champs en mer représentent 22 % de la production "
                                   "pétrolière nationale, contre 18 % cinq ans plus tôt."),
            dict(kind="bars", caption="Production par type de champ (millions de tonnes)",
                 labels=["Terrestre", "En mer", "Gaz assoc."], values=[70.6, 19.9, 12.4]),
        ],
        footer="Ministère de l'Énergie",
    ),
    dict(
        id="invoice_2481", lang="en", title="Invoice INV-2481",
        subtitle="Northwind Analytics Ltd — billing statement",
        fact="Invoice INV-2481 totals EUR 14,820.00 and is due within 30 days.",
        blocks=[
            dict(kind="table", header=["Item", "Qty", "Unit", "Amount"],
                 rows=[["Data platform licence", "12", "850.00", "10,200.00"],
                       ["Onboarding workshop", "2", "1,400.00", "2,800.00"],
                       ["Priority support", "1", "1,820.00", "1,820.00"]]),
            dict(kind="text", text="Total due: EUR 14,820.00. Payment terms: net 30 days "
                                   "from the invoice date. Late payment attracts 2% monthly interest."),
        ],
        footer="Northwind Analytics Ltd — VAT IE7412589X",
    ),
    dict(
        id="coffee_sales", lang="es", title="Ventas trimestrales de café",
        subtitle="Informe comercial 2024",
        fact="Q3 2024 coffee sales reached 8,900 units, the highest quarter of the year.",
        blocks=[
            dict(kind="heading", text="Unidades vendidas por trimestre"),
            dict(kind="bars", caption="Miles de unidades",
                 labels=["T1", "T2", "T3", "T4"], values=[6.2, 7.4, 8.9, 8.1]),
            dict(kind="text", text="El tercer trimestre fue el más fuerte del año, con 8.900 "
                                   "unidades vendidas, impulsado por la campaña de verano."),
        ],
        footer="Departamento comercial",
    ),
    dict(
        id="rainfall_de", lang="de", title="Niederschlagsstatistik",
        subtitle="Monatliche Messwerte, Station Freiburg",
        fact="The wettest month in Freiburg was July with 142 mm of rainfall.",
        blocks=[
            dict(kind="table", header=["Monat", "Niederschlag (mm)", "Regentage"],
                 rows=[["Mai", "98", "13"], ["Juni", "121", "15"],
                       ["Juli", "142", "16"], ["August", "115", "12"],
                       ["September", "87", "10"]]),
            dict(kind="text", text="Der Juli war mit 142 mm der niederschlagsreichste Monat "
                                   "des Beobachtungszeitraums."),
        ],
        footer="Deutscher Wetterdienst",
    ),
    dict(
        id="hr_parental", lang="en", title="Employee Handbook",
        subtitle="Section 7 — Family leave",
        fact="Parental leave is 16 weeks at full pay, extendable by 8 unpaid weeks.",
        blocks=[
            dict(kind="heading", text="7.3 Parental leave"),
            dict(kind="text", text="All employees with more than six months of service are entitled "
                                   "to 16 weeks of parental leave at full pay. A further 8 weeks may "
                                   "be taken unpaid, subject to two months' written notice."),
            dict(kind="heading", text="7.4 Carer's leave"),
            dict(kind="text", text="Up to 5 paid days per calendar year may be taken to care for "
                                   "a dependent relative."),
        ],
        footer="People Operations — revision 12",
    ),
    dict(
        id="battery_cycles", lang="en", title="Battery Ageing Study",
        subtitle="Capacity retention across charge cycles",
        fact="Cell capacity retention falls to 81% after 1000 charge cycles.",
        blocks=[
            dict(kind="heading", text="Retention vs. cycle count"),
            dict(kind="bars", caption="Capacity retention (%)",
                 labels=["0", "250", "500", "750", "1000"], values=[100, 96, 91, 86, 81]),
            dict(kind="text", text="Retention degraded roughly linearly, reaching 81 percent "
                                   "of nominal capacity after one thousand cycles at 25 degrees C."),
        ],
        footer="Materials Lab — internal report",
    ),
    dict(
        id="train_it", lang="it", title="Orario ferroviario",
        subtitle="Linea Milano — Bologna",
        fact="The fastest Milan to Bologna train takes 65 minutes.",
        blocks=[
            dict(kind="table", header=["Treno", "Partenza", "Arrivo", "Durata"],
                 rows=[["FR 9612", "06:20", "07:25", "65 min"],
                       ["IC 1534", "07:05", "08:40", "95 min"],
                       ["FR 9620", "08:20", "09:27", "67 min"],
                       ["RV 2210", "09:15", "11:10", "115 min"]]),
            dict(kind="text", text="Il collegamento più rapido impiega 65 minuti."),
        ],
        footer="Trenitalia — orario invernale",
    ),
    dict(
        id="trial_pt", lang="pt", title="Ensaio clínico fase II",
        subtitle="Resultados preliminares",
        fact="The treatment arm showed a 34% response rate versus 12% for placebo.",
        blocks=[
            dict(kind="table", header=["Grupo", "N", "Resposta", "Taxa"],
                 rows=[["Tratamento", "148", "50", "34%"],
                       ["Placebo", "150", "18", "12%"]]),
            dict(kind="text", text="A taxa de resposta no grupo de tratamento foi de 34 por cento, "
                                   "contra 12 por cento no grupo placebo."),
        ],
        footer="Centro de Investigação Clínica",
    ),
    dict(
        id="energy_mix", lang="en", title="Energy Mix 2025",
        subtitle="Share of generation by source",
        fact="Wind provided 34% of generation in 2025, the largest single source.",
        blocks=[
            dict(kind="bars", caption="Share of total generation (%)",
                 labels=["Wind", "Gas", "Solar", "Nuclear", "Hydro"],
                 values=[34, 27, 18, 14, 7]),
            dict(kind="text", text="Wind overtook gas for the first time, supplying 34 percent "
                                   "of all electricity generated during the year."),
        ],
        footer="Energy Regulator — annual review",
    ),
    dict(
        id="budget_fr", lang="fr", title="Budget prévisionnel 2025",
        subtitle="Répartition par département",
        fact="The R&D department receives EUR 4.6 million, the largest budget line.",
        blocks=[
            dict(kind="bars", caption="Budget en millions d'euros",
                 labels=["R&D", "Ventes", "Support", "RH", "IT"],
                 values=[4.6, 3.2, 1.8, 1.1, 2.4]),
            dict(kind="text", text="La recherche et développement reçoit la dotation la plus "
                                   "importante, soit 4,6 millions d'euros."),
        ],
        footer="Direction financière",
    ),
    dict(
        id="safety_es", lang="es", title="Normas de seguridad",
        subtitle="Almacén central — instrucciones",
        fact="Warehouse forklifts have a maximum speed limit of 8 km/h.",
        blocks=[
            dict(kind="heading", text="Circulación de carretillas"),
            dict(kind="text", text="La velocidad máxima permitida para las carretillas elevadoras "
                                   "dentro del almacén es de 8 km/h. Está prohibido transportar "
                                   "personas en la horquilla."),
            dict(kind="heading", text="Equipo de protección"),
            dict(kind="text", text="El uso de casco y chaleco reflectante es obligatorio en toda "
                                   "la zona de carga."),
        ],
        footer="Prevención de riesgos laborales",
    ),
]

CORPUS = [render_page(spec) for spec in CORPUS_SPECS]
DOC_IDS = [spec["id"] for spec in CORPUS_SPECS]

print(f"rendered {len(CORPUS)} pages at {CORPUS[0].size[0]}x{CORPUS[0].size[1]} px")
print("languages:", sorted({spec['lang'] for spec in CORPUS_SPECS}))
show_images(CORPUS[:4], [f"{s['id']} [{s['lang']}]" for s in CORPUS_SPECS[:4]], cols=4, width=3.0)
show_images(CORPUS[4:8], [f"{s['id']} [{s['lang']}]" for s in CORPUS_SPECS[4:8]], cols=4, width=3.0)
show_images(CORPUS[8:], [f"{s['id']} [{s['lang']}]" for s in CORPUS_SPECS[8:]], cols=4, width=3.0)

### 6.3 The evaluation set

Twenty queries with a gold page each. Deliberately mixed:

* **monolingual** — English query, English page;
* **cross-lingual** — English query, French/Spanish/German page (and vice versa);
* **visual** — the answer is only in a bar chart or table, never in prose;
* **distractor-prone** — several queries mention "energy" or "2025", but only one page answers each.

In [ ]:
@dataclass
class Query:
    text: str
    gold: str            # document id that answers it
    lang: str            # language of the query
    kind: str            # mono | cross | visual


QUERIES = [
    Query("Which hour of the day had the highest overall electricity generation in 2019?", "energy_hourly", "en", "visual"),
    Query("What share of Kazakhstan's oil production comes from offshore fields?", "kazakhstan_oil", "en", "cross"),
    Query("Quelle partie de la production pétrolière du Kazakhstan provient de champs en mer ?", "kazakhstan_oil", "fr", "mono"),
    Query("What is the total amount due on invoice INV-2481?", "invoice_2481", "en", "visual"),
    Query("How many days do I have to pay this invoice?", "invoice_2481", "en", "mono"),
    Query("¿Cuántas unidades de café se vendieron en el tercer trimestre?", "coffee_sales", "es", "mono"),
    Query("Which quarter had the strongest coffee sales?", "coffee_sales", "en", "cross"),
    Query("Welcher Monat hatte den meisten Niederschlag in Freiburg?", "rainfall_de", "de", "mono"),
    Query("How much rainfall did the wettest month record?", "rainfall_de", "en", "cross"),
    Query("How many weeks of paid parental leave do employees get?", "hr_parental", "en", "mono"),
    Query("Can I take unpaid leave to care for a relative?", "hr_parental", "en", "mono"),
    Query("What is the battery capacity retention after 1000 charge cycles?", "battery_cycles", "en", "visual"),
    Query("Qual è il treno più veloce da Milano a Bologna?", "train_it", "it", "mono"),
    Query("How long does the fastest train from Milan to Bologna take?", "train_it", "en", "cross"),
    Query("Qual foi a taxa de resposta no grupo de tratamento?", "trial_pt", "pt", "mono"),
    Query("Which energy source supplied the largest share of generation in 2025?", "energy_mix", "en", "visual"),
    Query("Quel département reçoit le budget le plus important ?", "budget_fr", "fr", "mono"),
    Query("Which department has the biggest budget allocation?", "budget_fr", "en", "cross"),
    Query("¿Cuál es la velocidad máxima de las carretillas en el almacén?", "safety_es", "es", "mono"),
    Query("What is the forklift speed limit in the warehouse?", "safety_es", "en", "cross"),
]

from collections import Counter
print(f"{len(QUERIES)} queries")
print("  by kind:", dict(Counter(q.kind for q in QUERIES)))
print("  by language:", dict(Counter(q.lang for q in QUERIES)))
assert {q.gold for q in QUERIES} <= set(DOC_IDS), "a query points at a page that does not exist"
print("\n✅ every gold label resolves to a page in the corpus")

<a name="s7"></a>
## 7. Retrieval with `NeoMMEForRetrieval`

`NeoMME-260M-Retriever` is the backbone fine-tuned for visual document retrieval with a **joint objective**:
a late-interaction loss on the token embeddings and a dense loss on the mean-pooled vector, trained together.
That is why one forward pass gives you both.

> **Note:** `NeoMMEForRetrieval` is not registered under a generic `AutoModelFor…` class. Import it
> directly — `from transformers import NeoMMEForRetrieval` — as the official docs do.

### 7.1 Load the retriever

In [ ]:
from transformers import NeoMMEForRetrieval, NeoMMEProcessor

retriever_processor = NeoMMEProcessor.from_pretrained(RETRIEVER_ID)
retriever = NeoMMEForRetrieval.from_pretrained(RETRIEVER_ID, dtype=DTYPE).to(DEVICE).eval()

print(f"{type(retriever).__name__}: {param_count(retriever):,} params")
print(f"late-interaction dim (config.embedding_dim): {retriever.config.embedding_dim}")
print(f"dense dim (config.hidden_size)             : {retriever.config.hidden_size}")
print(f"\nheads: multi_vector_head={type(retriever.multi_vector_head).__name__}, "
      f"dense_head={type(retriever.dense_head).__name__}")

### 7.2 The chat template: queries and documents are formatted differently

This is the part people get wrong. A query and a document are **not** encoded the same way, and you must
say which is which. `apply_chat_template(..., task=...)` handles the formatting:

| `task` | Rendered as | Why |
|---|---|---|
| `"document"` | `<doc>` + text, or `<doc>` + the `<img>` patch grid | Marks content to be indexed |
| `"query"` | `<query>` + text + **10 `<mask>` tokens** | Query expansion — see below |

Those ten trailing `<mask>` tokens are **query expansion**, the same trick ColBERT uses. They are not
padding: the model attends to the query through them and writes ten extra "what else might this question
be asking?" vectors into the multi-vector representation. They give a short query enough surface area to
match a whole page. The processor will raise if truncation eats them.

Let's look at the actual token stream.

In [ ]:
rtok = retriever_processor.tokenizer


def encode_batch(messages, task):
    """The canonical NeoMME retrieval encoding call."""
    return retriever_processor.apply_chat_template(
        messages,
        task=task,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"padding": "longest"},
    )


query_text = "Which quarter had the strongest coffee sales?"
q_inputs = encode_batch([[{"role": "user", "content": query_text}]], "query")
d_inputs = encode_batch([[{"role": "user", "content": [{"type": "text", "text": query_text}]}]], "document")

for label, inputs in (("task='query'   ", q_inputs), ("task='document'", d_inputs)):
    ids = inputs["input_ids"][0]
    real = ids[inputs["attention_mask"][0].bool()]
    pieces = rtok.convert_ids_to_tokens(real.tolist())
    print(f"{label}  {len(real)} tokens")
    print(f"                 {' '.join(repr(p) for p in pieces)}\n")

### 7.3 Encoding a page image

Same call, `task="document"`, with image content. Three constraints the template enforces — all of them
will raise a clear exception rather than silently misbehave:

1. **One image per conversation.** A page is the unit of retrieval.
2. **No text and image in the same conversation.** Encode the caption as its own document if you need it.
3. **Images require `task="document"`.** You cannot search *with* an image using this template.

In [ ]:
page_messages = [[{"role": "user", "content": [{"type": "image", "image": CORPUS[3]}]}]]
page_inputs = encode_batch(page_messages, "document")

print("encoded one page:")
for key, value in page_inputs.items():
    print(f"  {key:<15} {tuple(value.shape)}")

with torch.inference_mode():
    page_out = retriever(**page_inputs.to(DEVICE))

print(f"\nembeddings       {tuple(page_out.embeddings.shape)}       "
      f"<- (batch, seq_len, {retriever.config.embedding_dim}) late interaction")
print(f"dense_embeddings {tuple(page_out.dense_embeddings.shape)}            "
      f"<- (batch, {retriever.config.hidden_size}) mean-pooled")
print(f"last_hidden_state{tuple(page_out.last_hidden_state.shape)}     <- backbone states, still there")

# Two properties the heads guarantee, worth checking once.
real = page_out.embeddings[page_inputs["attention_mask"].bool()]
print(f"\ntoken embedding L2 norms: min={float(real.norm(dim=-1).min()):.4f} "
      f"max={float(real.norm(dim=-1).max()):.4f}   (normalized)")
padding = page_out.embeddings[~page_inputs["attention_mask"].bool()]
print(f"padding rows: {padding.shape[0]} rows, max abs value "
      f"{float(padding.abs().max()) if padding.numel() else 0.0}   (exactly zero)")

Both guarantees matter downstream. Normalized token vectors mean a dot product **is** cosine similarity,
so MaxSim is just an `einsum`. Zeroed padding rows mean you can throw the whole padded tensor at a scoring
function without a mask and still get roughly the right answer (though passing the mask is better —
a zero row still scores 0, which can beat a genuinely negative similarity).

### 7.4 Try it on the guard rails

Failing loudly beats failing quietly. Confirm the template rejects malformed input.

In [ ]:
bad_cases = [
    ("image with task='query'",
     lambda: encode_batch(page_messages, "query")),
    ("text and image in one conversation",
     lambda: encode_batch([[{"role": "user", "content": [
         {"type": "text", "text": "caption"},
         {"type": "image", "image": CORPUS[0]}]}]], "document")),
    ("two images in one conversation",
     lambda: encode_batch([[{"role": "user", "content": [
         {"type": "image", "image": CORPUS[0]},
         {"type": "image", "image": CORPUS[1]}]}]], "document")),
    ("no task at all",
     lambda: retriever_processor.apply_chat_template(
         [[{"role": "user", "content": "hello"}]], tokenize=True, return_dict=True,
         return_tensors="pt")),
    ("truncation that eats the query expansion masks",
     lambda: retriever_processor.apply_chat_template(
         [[{"role": "user", "content": "a fairly long question about coffee sales"}]],
         task="query", tokenize=True, return_dict=True, return_tensors="pt",
         processor_kwargs={"max_length": 5, "truncation": True})),
]

for label, thunk in bad_cases:
    try:
        thunk()
        print(f"  ⚠️  {label}: no error raised")
    except Exception as exc:
        message = str(exc).split("\n")[0][:88]
        print(f"  ✅ {label}\n      -> {type(exc).__name__}: {message}")

### 7.5 Two ways to score

Now the payoff. We have two representations of every input; here are the two ways to compare them.

**Dense — `cos_sim(query_dense, doc_dense)`.** One 1024-d vector per page. A single matrix multiply scores
a million pages. This is your first-stage retriever.

**Late interaction — `mean_maxsim(query_tokens, doc_tokens)`.** For each *query* token, find its best-matching
*document* token, then average those maxima over the query. Far more expressive — a query token for
"offshore" can light up one specific patch in the middle of a chart — but you store hundreds of vectors per
page. This is your reranker.

> **Why *Mean*MaxSim and not plain MaxSim?** Plain MaxSim *sums* over query tokens, so its scale grows with
> query length and scores are not comparable across queries — a threshold that works for a 5-token query is
> meaningless for a 20-token one. MeanMaxSim divides by the query's real token count, landing back in the
> per-token similarity range of roughly `[-1, 1]`. Rankings *within* a query are identical (the divisor is
> constant per row); what you gain is comparability *across* queries, which is what you need for score
> thresholds, fusion, and calibration.

In [ ]:
from sentence_transformers.util import cos_sim, maxsim, mean_maxsim

demo_queries = [
    "Which quarter had the strongest coffee sales?",
    "What is the forklift speed limit in the warehouse?",
]
demo_pages = [CORPUS[DOC_IDS.index(i)] for i in ("coffee_sales", "safety_es", "invoice_2481")]
demo_ids = ["coffee_sales", "safety_es", "invoice_2481"]

q_in = encode_batch([[{"role": "user", "content": q}] for q in demo_queries], "query").to(DEVICE)
d_in = encode_batch([[{"role": "user", "content": [{"type": "image", "image": p}]}] for p in demo_pages],
                    "document").to(DEVICE)

with torch.inference_mode():
    q_out = retriever(**q_in)
    d_out = retriever(**d_in)

late = mean_maxsim(q_out.embeddings.float(), d_out.embeddings.float(),
                   a_mask=q_in["attention_mask"], b_mask=d_in["attention_mask"])
dense = cos_sim(q_out.dense_embeddings.float(), d_out.dense_embeddings.float())
plain = maxsim(q_out.embeddings.float(), d_out.embeddings.float(),
               a_mask=q_in["attention_mask"], b_mask=d_in["attention_mask"])

header = f"{'':<52}" + "".join(f"{d[:14]:>16}" for d in demo_ids)
for name, matrix in (("MeanMaxSim (late interaction)", late),
                     ("MaxSim (unnormalized)", plain),
                     ("cosine (dense)", dense)):
    print(f"\n{name}")
    print(header)
    for row, query in enumerate(demo_queries):
        cells = "".join(f"{float(matrix[row, col]):>16.4f}" for col in range(len(demo_ids)))
        best = "  ✅" if int(matrix[row].argmax()) == row else "  ❌"
        print(f"  {query[:50]:<50}{cells}{best}")

n_query_tokens = q_in["attention_mask"].sum(dim=1)
print(f"\nquery token counts: {n_query_tokens.tolist()}")
print(f"MaxSim / MeanMaxSim ratio: {(plain / late)[:, 0].tolist()}  <- exactly the token counts")

### 7.6 What is the late-interaction head actually looking at?

Late interaction is interpretable in a way dense embeddings are not: for each query token you can point at
the **exact image patch** it matched. Let's draw that heatmap over the page.

In [ ]:
def maxsim_heatmap(query, page, patch_size=None):
    """Per-patch max similarity to any query token, reshaped to the page's patch grid."""
    patch_size = patch_size or retriever.config.patch_size

    qi = encode_batch([[{"role": "user", "content": query}]], "query").to(DEVICE)
    di = encode_batch([[{"role": "user", "content": [{"type": "image", "image": page}]}]],
                      "document").to(DEVICE)
    with torch.inference_mode():
        qe = retriever(**qi).embeddings[0].float()                # (q_len, 128)
        de = retriever(**di).embeddings[0].float()                # (d_len, 128)

    similarity = qe @ de.T                                        # (q_len, d_len)
    per_document_token = similarity.max(dim=0).values             # best query match per doc token

    # Rebuild the grid: the sequence is <doc> <img> then rows of (grid_w patches + 1 <row>).
    ids = di["input_ids"][0]
    grid_w = int((ids[2:] == rtok.row_token_id).nonzero()[0]) if (ids == rtok.row_token_id).any() else None
    grid_h = int((ids == rtok.row_token_id).sum())

    scores = per_document_token[2:].reshape(grid_h, grid_w + 1)[:, :grid_w]   # drop <row> column
    return scores.cpu().numpy(), (grid_h, grid_w)


query = "Which quarter had the strongest coffee sales?"
page = CORPUS[DOC_IDS.index("coffee_sales")]
heat, (gh, gw) = maxsim_heatmap(query, page)

fig, axes = plt.subplots(1, 3, figsize=(14, 6))
axes[0].imshow(page); axes[0].set_title("page"); axes[0].axis("off")
im = axes[1].imshow(heat, cmap="inferno")
axes[1].set_title(f"MaxSim heat ({gh}x{gw} patches)"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046)

overlay = np.array(page.resize((gw * 8, gh * 8))).astype(float) / 255
heat_norm = (heat - heat.min()) / (np.ptp(heat) + 1e-9)
heat_big = np.kron(heat_norm, np.ones((8, 8)))[..., None]
axes[2].imshow(np.clip(overlay * 0.35 + heat_big * np.array([1.0, 0.35, 0.1]) * 0.9, 0, 1))
axes[2].set_title("overlay"); axes[2].axis("off")
plt.suptitle(f'"{query}"', fontsize=11)
plt.tight_layout()
plt.show()

top = np.dstack(np.unravel_index(np.argsort(heat.ravel())[::-1][:5], heat.shape))[0]
print("hottest patches (row, col):", [tuple(map(int, rc)) for rc in top])

The bright regions are the patches the query tokens latched onto — typically the chart bars and the
surrounding numerals, not the header band. That is the single-tower model *reading the chart*, with no OCR
step anywhere in the pipeline.

<a name="s8"></a>
## 8. Building — and measuring — a real search engine

Time to stop poking at individual tensors and build the thing. Index all 12 pages, run all 20 queries,
and compute honest IR metrics for three strategies: dense-only, late-interaction-only, and the two-stage
pipeline you would actually ship.

### 8.1 The index

A small, explicit class. Documents of different sizes produce different token counts, so we keep the
multi-vector representation as a **list of variable-length tensors** rather than one padded block — that
is both what `mean_maxsim` accepts and what a real vector store would hold.

In [ ]:
@dataclass
class Index:
    doc_ids: list
    dense: torch.Tensor              # (num_docs, hidden_size)
    multi: list                      # list of (num_tokens_i, embedding_dim)

    @property
    def num_docs(self):
        return len(self.doc_ids)

    def token_counts(self):
        return [int(t.shape[0]) for t in self.multi]

    def nbytes(self, dtype_bytes=4):
        dense = self.dense.numel() * dtype_bytes
        multi = sum(t.numel() for t in self.multi) * dtype_bytes
        return dense, multi


@torch.inference_mode()
def encode_documents(images, doc_ids, batch_size=4, keep_on="cpu", **image_kwargs):
    """Encode page images into dense + multi-vector representations."""
    dense_chunks, multi = [], []
    for start in range(0, len(images), batch_size):
        chunk = images[start : start + batch_size]
        messages = [[{"role": "user", "content": [{"type": "image", "image": image}]}] for image in chunk]
        inputs = retriever_processor.apply_chat_template(
            messages, task="document", tokenize=True, return_dict=True, return_tensors="pt",
            processor_kwargs={"padding": "longest", **image_kwargs},
        ).to(DEVICE)

        out = retriever(**inputs)
        dense_chunks.append(out.dense_embeddings.float().to(keep_on))
        mask = inputs["attention_mask"].bool()
        for row in range(len(chunk)):
            multi.append(out.embeddings[row][mask[row]].float().to(keep_on))

    return Index(list(doc_ids), torch.cat(dense_chunks), multi)


@torch.inference_mode()
def encode_queries(texts, batch_size=16, keep_on="cpu"):
    """Encode query strings. Returns (dense (n, H), list of (tokens_i, D))."""
    dense_chunks, multi = [], []
    for start in range(0, len(texts), batch_size):
        chunk = texts[start : start + batch_size]
        inputs = retriever_processor.apply_chat_template(
            [[{"role": "user", "content": text}] for text in chunk],
            task="query", tokenize=True, return_dict=True, return_tensors="pt",
            processor_kwargs={"padding": "longest"},
        ).to(DEVICE)

        out = retriever(**inputs)
        dense_chunks.append(out.dense_embeddings.float().to(keep_on))
        mask = inputs["attention_mask"].bool()
        for row in range(len(chunk)):
            multi.append(out.embeddings[row][mask[row]].float().to(keep_on))

    return torch.cat(dense_chunks), multi


with Timer(f"indexing {len(CORPUS)} pages"):
    index = encode_documents(CORPUS, DOC_IDS, batch_size=4)

with Timer(f"encoding {len(QUERIES)} queries"):
    query_dense, query_multi = encode_queries([q.text for q in QUERIES])

counts = index.token_counts()
dense_bytes, multi_bytes = index.nbytes()
print(f"\nindexed {index.num_docs} pages")
print(f"  tokens per page : min={min(counts)} max={max(counts)} mean={sum(counts) / len(counts):.0f}")
print(f"  dense index     : {tuple(index.dense.shape)}  {human_bytes(dense_bytes)}")
print(f"  multi index     : {sum(counts):,} vectors x {index.multi[0].shape[1]}  {human_bytes(multi_bytes)}")
print(f"  ratio           : late interaction is {multi_bytes / dense_bytes:.0f}x larger")

That ratio is the whole trade-off in one number. Late interaction typically costs **two to three orders of
magnitude** more storage than a dense index. Section 9 claws most of it back.

### 8.2 Metrics

Standard IR metrics, implemented from scratch so there is nothing to take on trust. Each query has exactly
one relevant document, which simplifies things: nDCG@k reduces to `1 / log2(rank + 1)` when the gold
document is in the top k, and 0 otherwise.

In [ ]:
def evaluate(score_matrix, queries, doc_ids, k_values=(1, 3, 5, 10)):
    """Recall@k, MRR and nDCG@k for a (num_queries, num_docs) score matrix."""
    ranking = torch.as_tensor(score_matrix).argsort(dim=1, descending=True)
    gold_positions = []
    for row, query in enumerate(queries):
        gold_column = doc_ids.index(query.gold)
        rank = int((ranking[row] == gold_column).nonzero()) + 1     # 1-based
        gold_positions.append(rank)

    metrics = {"MRR": float(np.mean([1.0 / r for r in gold_positions]))}
    for k in k_values:
        metrics[f"Recall@{k}"] = float(np.mean([r <= k for r in gold_positions]))
        metrics[f"nDCG@{k}"] = float(np.mean([1.0 / np.log2(r + 1) if r <= k else 0.0
                                              for r in gold_positions]))
    return metrics, gold_positions


def report(title, metrics, extra=""):
    keys = ["Recall@1", "Recall@3", "Recall@5", "MRR", "nDCG@5", "nDCG@10"]
    cells = "  ".join(f"{key}={metrics[key]:.3f}" for key in keys)
    print(f"{title:<34} {cells} {extra}")


print("✅ metrics ready")

### 8.3 Strategy 1 — dense retrieval

One vector per page, one matrix multiply. This is what a classic embedding model gives you.

In [ ]:
with Timer("dense scoring"):
    dense_scores = cos_sim(query_dense, index.dense)

dense_metrics, dense_ranks = evaluate(dense_scores, QUERIES, index.doc_ids)
report("dense (cosine, 1024-d)", dense_metrics)

### 8.4 Strategy 2 — late interaction

In [ ]:
with Timer("MeanMaxSim scoring"):
    late_scores = mean_maxsim(query_multi, index.multi)

late_metrics, late_ranks = evaluate(late_scores, QUERIES, index.doc_ids)
report("late interaction (MeanMaxSim)", late_metrics)

print(f"\nlate interaction is {multi_bytes / dense_bytes:.0f}x the storage and "
      f"{late_metrics['Recall@1'] - dense_metrics['Recall@1']:+.3f} Recall@1")

### 8.5 Strategy 3 — the two-stage pipeline you should actually ship

Dense retrieval sweeps the whole corpus cheaply; late interaction reranks the survivors. With
`rerank_depth = 5` on a 12-page corpus this is a toy, but the shape is exactly what scales: dense over
10M pages in an ANN index, MeanMaxSim over the top 200.

The cost of the pipeline is the cost of the dense stage plus `rerank_depth` MaxSim computations — and the
crucial property is that **the multi-vector index only needs to be readable for candidates**, so it can
live on disk or in a cheaper store.

In [ ]:
def two_stage_search(query_dense_vecs, query_multi_vecs, index, rerank_depth=5):
    """Dense first-stage sweep, MeanMaxSim rerank of the top `rerank_depth`."""
    first_stage = cos_sim(query_dense_vecs, index.dense)
    depth = min(rerank_depth, index.num_docs)
    candidates = first_stage.argsort(dim=1, descending=True)[:, :depth]

    # Start from the dense scores, then overwrite the reranked candidates with a boosted score so
    # reranked documents always outrank un-reranked ones (scores live on different scales).
    final = first_stage.clone() - 10.0
    for row in range(len(query_multi_vecs)):
        columns = candidates[row].tolist()
        rerank = mean_maxsim([query_multi_vecs[row]], [index.multi[c] for c in columns])[0]
        for position, column in enumerate(columns):
            final[row, column] = float(rerank[position])
    return final


for depth in (3, 5, 8, index.num_docs):
    scores = two_stage_search(query_dense, query_multi, index, rerank_depth=depth)
    metrics, _ = evaluate(scores, QUERIES, index.doc_ids)
    report(f"two-stage (rerank top {depth})", metrics)

print()
report("dense only", dense_metrics)
report("late interaction only", late_metrics)

### 8.6 Where does it fail, and why?

Aggregate metrics hide the interesting cases. Let's look at every query individually, split by kind, and
find the ones where the two strategies disagree.

In [ ]:
print(f"{'kind':<7}{'lang':<6}{'query':<58}{'dense':>7}{'late':>7}")
print("-" * 86)
for row, query in enumerate(QUERIES):
    d_rank, l_rank = dense_ranks[row], late_ranks[row]
    mark = lambda r: f"{r}" + ("" if r == 1 else " ")
    flag = "" if d_rank == l_rank else ("   ← late wins" if l_rank < d_rank else "   ← dense wins")
    print(f"{query.kind:<7}{query.lang:<6}{query.text[:56]:<58}{d_rank:>7}{l_rank:>7}{flag}")

print("\nby query kind (mean reciprocal rank):")
for kind in ("mono", "cross", "visual"):
    rows = [i for i, q in enumerate(QUERIES) if q.kind == kind]
    d_mrr = np.mean([1 / dense_ranks[i] for i in rows])
    l_mrr = np.mean([1 / late_ranks[i] for i in rows])
    print(f"  {kind:<8} n={len(rows):<3} dense MRR={d_mrr:.3f}   late MRR={l_mrr:.3f}")

print("\nby query language:")
for lang in sorted({q.lang for q in QUERIES}):
    rows = [i for i, q in enumerate(QUERIES) if q.lang == lang]
    d_mrr = np.mean([1 / dense_ranks[i] for i in rows])
    l_mrr = np.mean([1 / late_ranks[i] for i in rows])
    print(f"  {lang:<8} n={len(rows):<3} dense MRR={d_mrr:.3f}   late MRR={l_mrr:.3f}")

The `cross` row is the one to dwell on. Those are queries asked in English against pages written in French,
Spanish, German, Italian or Portuguese. There is **no translation step** anywhere in this notebook — the
multilingual vocabulary and the shared encoder do the work, and the query never sees anything but pixels
on the document side.

### 8.7 Show the results

Finally, the thing an end user would see.

In [ ]:
def search(query_text, top_k=3, rerank_depth=5, show=True):
    """End-to-end: text query in, ranked pages out."""
    q_dense, q_multi = encode_queries([query_text])
    scores = two_stage_search(q_dense, q_multi, index, rerank_depth=rerank_depth)[0]
    order = scores.argsort(descending=True)[:top_k].tolist()
    hits = [(index.doc_ids[i], float(scores[i]), CORPUS[i]) for i in order]

    if show:
        print(f'🔍  "{query_text}"')
        for rank, (doc_id, score, _) in enumerate(hits, 1):
            fact = next(s["fact"] for s in CORPUS_SPECS if s["id"] == doc_id)
            print(f"   {rank}. {doc_id:<18} score={score:6.3f}   {fact}")
        show_images([h[2] for h in hits],
                    [f"#{r} {h[0]} ({h[1]:.3f})" for r, h in enumerate(hits, 1)],
                    cols=top_k, width=3.0)
    return hits


search("Which energy source supplied the largest share of generation in 2025?")
search("Combien de semaines de congé parental sont payées ?")   # French query, English page

<a name="s9"></a>
## 9. Making the index small: Matryoshka, pooling, quantization

Section 8 ended on an uncomfortable number: the late-interaction index was two to three orders of magnitude
larger than the dense one. At 12 pages nobody cares. At 10 million pages — roughly 1,100 vectors per page,
128 dimensions, 4 bytes each — that is **~5.6 TB**. You cannot serve that.

NeoMME's headline efficiency claim is that hierarchical token pooling plus asymmetric quantization compress
late-interaction document embeddings by up to **255×** while preserving **over 95%** of baseline nDCG@10.
This section works through the three independent levers that get you there, measuring quality at each step
on our own corpus.

| Lever | Applies to | Mechanism | Typical saving |
|---|---|---|---|
| Matryoshka `dense_dim` | dense vectors | truncate the pooled vector before normalizing | 2–8× |
| Hierarchical token pooling | multi-vectors | cluster similar token vectors, keep the means | 2–64× |
| Quantization | both | int8 or binary instead of float32 | 4× / 32× |

They multiply. And they apply **only to the document side** — queries stay full precision, which is what
"asymmetric" means and why the quality loss is so small.

### 9.1 Matryoshka: dense vectors that truncate gracefully

`NeoMMEForRetrieval.forward(dense_dim=N)` slices the pooled vector to its first `N` dimensions *before*
normalizing. This only works because the model was trained so that the leading dimensions carry the most
information — you cannot do this to an arbitrary embedding model.

`dense_dim` must be between 1 and `hidden_size`.

In [ ]:
@torch.inference_mode()
def encode_dense_only(images, dense_dim=None, batch_size=4):
    chunks = []
    for start in range(0, len(images), batch_size):
        chunk = images[start : start + batch_size]
        inputs = retriever_processor.apply_chat_template(
            [[{"role": "user", "content": [{"type": "image", "image": i}]}] for i in chunk],
            task="document", tokenize=True, return_dict=True, return_tensors="pt",
            processor_kwargs={"padding": "longest"},
        ).to(DEVICE)
        out = retriever(**inputs, output_multivector=False, dense_dim=dense_dim)
        chunks.append(out.dense_embeddings.float().cpu())
    return torch.cat(chunks)


@torch.inference_mode()
def encode_query_dense_only(texts, dense_dim=None):
    inputs = retriever_processor.apply_chat_template(
        [[{"role": "user", "content": t}] for t in texts],
        task="query", tokenize=True, return_dict=True, return_tensors="pt",
        processor_kwargs={"padding": "longest"},
    ).to(DEVICE)
    return retriever(**inputs, output_multivector=False,
                     dense_dim=dense_dim).dense_embeddings.float().cpu()


hidden = retriever.config.hidden_size
dims = sorted({d for d in (64, 128, 256, 512, hidden) if d <= hidden})

print(f"{'dense_dim':>10}{'bytes/doc':>12}{'Recall@1':>11}{'MRR':>8}{'nDCG@5':>9}")
print("-" * 52)
matryoshka_rows = []
for dim in dims:
    doc_vectors = encode_dense_only(CORPUS, dense_dim=dim)
    query_vectors = encode_query_dense_only([q.text for q in QUERIES], dense_dim=dim)
    metrics, _ = evaluate(cos_sim(query_vectors, doc_vectors), QUERIES, DOC_IDS)
    matryoshka_rows.append((dim, metrics))
    print(f"{dim:>10}{dim * 4:>12,}{metrics['Recall@1']:>11.3f}"
          f"{metrics['MRR']:>8.3f}{metrics['nDCG@5']:>9.3f}")

full_ndcg = matryoshka_rows[-1][1]["nDCG@5"]
print(f"\nrelative to full {hidden}-d (nDCG@5 = {full_ndcg:.3f}):")
for dim, metrics in matryoshka_rows:
    keep = metrics["nDCG@5"] / full_ndcg
    bar = "█" * int(keep * 30)
    print(f"  {dim:>5}-d  {hidden // dim:>3}x smaller   {keep:>6.1%} of quality  {bar}")

### 9.2 Hierarchical token pooling

The insight: a page's token embeddings are enormously **redundant**. Whitespace patches all look alike; a
block of body text produces dozens of near-identical vectors. Ward-linkage hierarchical clustering on
cosine distance groups them and replaces each cluster with its mean.

`pool_factor=k` keeps roughly `1/k` of each document's tokens. `num_protected_tokens` leaves the leading
tokens untouched (they carry the `<doc>` marker's aggregate signal).

Documents only — never pool the query side. The query is a handful of vectors and each one is doing
specific work.

In [ ]:
from sentence_transformers.multi_vector_encoder.modules import HierarchicalTokenPooling

original_vectors = sum(t.shape[0] for t in index.multi)
baseline_late, _ = evaluate(mean_maxsim(query_multi, index.multi), QUERIES, DOC_IDS)

print(f"{'pool_factor':>12}{'vectors':>11}{'per doc':>10}{'index':>12}"
      f"{'Recall@1':>11}{'nDCG@5':>9}{'kept':>8}")
print("-" * 74)
print(f"{'1 (none)':>12}{original_vectors:>11,}{original_vectors / index.num_docs:>10.0f}"
      f"{human_bytes(original_vectors * 128 * 4):>12}"
      f"{baseline_late['Recall@1']:>11.3f}{baseline_late['nDCG@5']:>9.3f}{'100.0%':>8}")

pooled_indexes = {}
for factor in (2, 3, 4, 8, 16, 32):
    pooler = HierarchicalTokenPooling(pool_factor=factor, num_protected_tokens=1)
    with Timer() as t:
        pooled = pooler.pool([v for v in index.multi], task="document")
    pooled_indexes[factor] = pooled

    vectors = sum(p.shape[0] for p in pooled)
    metrics, _ = evaluate(mean_maxsim(query_multi, pooled), QUERIES, DOC_IDS)
    print(f"{factor:>12}{vectors:>11,}{vectors / index.num_docs:>10.0f}"
          f"{human_bytes(vectors * 128 * 4):>12}"
          f"{metrics['Recall@1']:>11.3f}{metrics['nDCG@5']:>9.3f}"
          f"{metrics['nDCG@5'] / baseline_late['nDCG@5']:>8.1%}")

Note the shape of that curve: quality is nearly flat for the first few pool factors and only then falls
away. That is the redundancy being squeezed out before any signal is. Where exactly your corpus falls off
depends on how dense your pages are — text-heavy pages tolerate more pooling than sparse diagrams.

### 9.3 Asymmetric quantization

The last lever. Store document vectors as **int8** with per-dimension calibrated ranges; keep query vectors
in **float32**. Dequantize on the fly (or use int8 kernels). "Asymmetric" is doing real work here: the
error introduced on the document side is averaged over hundreds of MaxSim comparisons, while the query side
— where a single bad vector is a single bad query — stays exact.

In [ ]:
def calibrate(vectors, percentile=99.9, max_sample=200_000):
    """Per-dimension ranges from a sample of document vectors.

    Percentiles rather than hard min/max, so one outlier component cannot stretch the
    range and waste quantization levels for every other vector.
    """
    stacked = torch.cat(vectors, dim=0)
    if stacked.shape[0] > max_sample:      # torch.quantile caps input size
        picks = torch.randperm(stacked.shape[0])[:max_sample]
        stacked = stacked[picks]
    low = torch.quantile(stacked, (100 - percentile) / 100, dim=0)
    high = torch.quantile(stacked, percentile / 100, dim=0)
    return low, high


def quantize_int8(vectors, low, high):
    """float32 -> int8 with per-dimension affine ranges."""
    scale = (high - low).clamp_min(1e-8) / 255.0
    return [((v - low) / scale - 128).round().clamp(-128, 127).to(torch.int8) for v in vectors], scale


def dequantize_int8(quantized, low, scale):
    return [(q.float() + 128) * scale + low for q in quantized]


low, high = calibrate(index.multi)
quantized, scale = quantize_int8(index.multi, low, high)
restored = dequantize_int8(quantized, low, scale)

error = torch.cat([(a - b).abs().flatten() for a, b in zip(index.multi, restored)])
q_metrics, _ = evaluate(mean_maxsim(query_multi, restored), QUERIES, DOC_IDS)

print(f"int8 asymmetric quantization")
print(f"  mean |error| per component : {float(error.mean()):.5f}")
print(f"  max  |error| per component : {float(error.max()):.5f}")
print(f"  storage                    : 4 bytes -> 1 byte per component (4x)")
print(f"  Recall@1  {baseline_late['Recall@1']:.3f} -> {q_metrics['Recall@1']:.3f}")
print(f"  nDCG@5    {baseline_late['nDCG@5']:.3f} -> {q_metrics['nDCG@5']:.3f}   "
      f"({q_metrics['nDCG@5'] / baseline_late['nDCG@5']:.1%} retained)")

Sentence Transformers ships a general-purpose helper for the same idea, which is what you'd use on the
**dense** index (where you can hand it a clean 2-D array):

In [ ]:
from sentence_transformers import quantize_embeddings

dense_np = index.dense.numpy()
for precision in ("float32", "int8", "binary"):
    if precision == "float32":
        quantized_dense = dense_np
    else:
        quantized_dense = quantize_embeddings(dense_np, precision=precision,
                                              calibration_embeddings=dense_np)
    print(f"  {precision:<9} shape={str(quantized_dense.shape):<12} "
          f"dtype={str(quantized_dense.dtype):<8} {human_bytes(quantized_dense.nbytes):>10} "
          f"({dense_np.nbytes / quantized_dense.nbytes:>4.0f}x)")

print("\n(binary packs 8 dimensions per byte — 32x smaller, scored with Hamming distance)")

### 9.4 Stacking all three

Pooling × quantization, measured end to end. This is the recipe behind the paper's headline number.

In [ ]:
print(f"{'configuration':<34}{'index size':>13}{'compression':>13}{'nDCG@5':>9}{'retained':>10}")
print("-" * 79)
baseline_bytes = original_vectors * 128 * 4
print(f"{'float32, no pooling (baseline)':<34}{human_bytes(baseline_bytes):>13}"
      f"{'1x':>13}{baseline_late['nDCG@5']:>9.3f}{'100.0%':>10}")

results = []
for factor in (1, 2, 3, 4, 8, 16, 32):
    pooled = index.multi if factor == 1 else pooled_indexes[factor]
    lo, hi = calibrate(pooled)
    q, sc = quantize_int8(pooled, lo, hi)
    deq = dequantize_int8(q, lo, sc)

    vectors = sum(p.shape[0] for p in pooled)
    size = vectors * 128 * 1                       # int8: one byte per component
    metrics, _ = evaluate(mean_maxsim(query_multi, deq), QUERIES, DOC_IDS)
    retained = metrics["nDCG@5"] / baseline_late["nDCG@5"]
    results.append((factor, size, metrics, retained))
    label = f"int8 + pool_factor={factor}"
    print(f"{label:<34}{human_bytes(size):>13}{baseline_bytes / size:>12.0f}x"
          f"{metrics['nDCG@5']:>9.3f}{retained:>10.1%}")

good = [r for r in results if r[3] >= 0.95]
if good:
    factor, size, metrics, retained = max(good, key=lambda r: baseline_bytes / r[1])
    print(f"\n🏆 best configuration keeping >=95% of nDCG@5: pool_factor={factor}, int8")
    print(f"   {baseline_bytes / size:.0f}x smaller, {retained:.1%} of baseline quality")
else:
    print("\n(no configuration cleared 95% on this 12-page corpus — the sample is tiny; "
          "the published result is 255x at >95% on ViDoRe v3)")

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot([baseline_bytes / r[1] for r in results], [r[3] for r in results], "o-", lw=2)
for factor, size, _, retained in results:
    ax.annotate(f"pf={factor}", (baseline_bytes / size, retained),
                textcoords="offset points", xytext=(6, -3), fontsize=8)
ax.axhline(0.95, color="crimson", ls="--", lw=1, label="95% of baseline nDCG@5")
ax.set_xscale("log"); ax.set_xlabel("compression vs. float32 unpooled (log scale)")
ax.set_ylabel("fraction of baseline nDCG@5"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Late-interaction index: size vs. quality")
plt.tight_layout(); plt.show()

> **Caveat, stated plainly.** Twelve pages and twenty queries is far too small a sample to reproduce a
> benchmark number — a single rank flip moves nDCG@5 by ~0.05. What this section *does* show reliably is
> the **shape** of the trade-off and the exact code to measure it on your own corpus. Run it on a few
> thousand of your real pages before choosing a `pool_factor`.

### 9.5 What this means at scale

Plugging the measured numbers into a realistic corpus:

In [ ]:
NUM_PAGES = 10_000_000
tokens_per_page = sum(index.token_counts()) / index.num_docs

print(f"corpus: {NUM_PAGES:,} pages, ~{tokens_per_page:.0f} tokens/page at "
      f"{CORPUS[0].size[0]}x{CORPUS[0].size[1]} px\n")

configs = [
    ("dense float32, 1024-d",             NUM_PAGES * 1024 * 4),
    ("dense int8, 256-d (Matryoshka)",    NUM_PAGES * 256 * 1),
    ("late float32, unpooled",            int(NUM_PAGES * tokens_per_page * 128 * 4)),
    ("late int8, pool_factor=4",          int(NUM_PAGES * tokens_per_page / 4 * 128 * 1)),
    ("late int8, pool_factor=16",         int(NUM_PAGES * tokens_per_page / 16 * 128 * 1)),
]
widest = max(len(name) for name, _ in configs)
for name, size in configs:
    print(f"  {name:<{widest}}  {human_bytes(size):>10}")

print("\nThe two-stage pipeline from 8.5 uses the second row for the sweep and the fourth for")
print("reranking only the top few hundred candidates — which is why it fits on one machine.")

<a name="s10"></a>
## 10. Fine-tuning NeoMME on your own data

The released retriever is strong on general visual documents. It has never seen *your* invoices, your
schematics, your internal jargon. Fine-tuning on a few thousand labelled query–page pairs is usually the
single highest-return thing you can do.

Three routes, in increasing order of convenience:

1. **Raw PyTorch contrastive training** — Section 10.2. Fully transparent, no extra dependencies, ~40 lines.
   Use this to understand what is happening and when you need an unusual loss.
2. **Sentence Transformers `MultiVectorEncoderTrainer`** — Section 10.4. Batteries included: hard-negative
   mining, cached losses, evaluators, checkpointing, model cards.
3. **Continued MLM pretraining first** — Section 10.5. If your domain vocabulary is genuinely alien
   (chemistry notation, legal boilerplate in a language the pretraining under-served), adapt the backbone
   before you fine-tune the retriever.

### 10.1 Build a training set

Labelled data for visual retrieval means `(query, page)` pairs. The standard trick — and it works
remarkably well — is to **generate queries from pages you already have**, then have humans filter. Here we
build pairs from the corpus facts, split by document so train and eval never share a page.

In [ ]:
TRAIN_PAIRS = [
    ("How much electricity was generated at 18:00?",              "energy_hourly"),
    ("Peak generation hour in the 2019 grid report",              "energy_hourly"),
    ("Offshore share of Kazakh oil output",                       "kazakhstan_oil"),
    ("Production totale de pétrole au Kazakhstan",                "kazakhstan_oil"),
    ("Amount payable on the Northwind invoice",                   "invoice_2481"),
    ("Payment terms net 30 days",                                 "invoice_2481"),
    ("Ventas de café por trimestre",                              "coffee_sales"),
    ("Best selling quarter for coffee",                           "coffee_sales"),
    ("Regenmenge im Juli in Freiburg",                            "rainfall_de"),
    ("Monthly rainfall table",                                    "rainfall_de"),
    ("Paid parental leave entitlement",                           "hr_parental"),
    ("Notice period for unpaid family leave",                     "hr_parental"),
]

EVAL_PAIRS = [
    ("Capacity retention after a thousand cycles",                "battery_cycles"),
    ("Battery degradation study results",                         "battery_cycles"),
    ("Fastest connection Milano Bologna",                         "train_it"),
    ("Response rate in the treatment group",                      "trial_pt"),
    ("Largest source of electricity in 2025",                     "energy_mix"),
    ("Budget de la recherche et développement",                   "budget_fr"),
    ("Maximum forklift speed in the warehouse",                   "safety_es"),
]

train_docs = sorted({doc for _, doc in TRAIN_PAIRS})
eval_docs = sorted({doc for _, doc in EVAL_PAIRS})
assert not set(train_docs) & set(eval_docs), "train/eval page leakage"

print(f"train: {len(TRAIN_PAIRS)} pairs over {len(train_docs)} pages")
print(f"eval : {len(EVAL_PAIRS)} pairs over {len(eval_docs)} pages")
print("✅ no page appears in both splits")

### 10.2 Contrastive fine-tuning in raw PyTorch

The loss is **in-batch multiple negatives** (InfoNCE): for a batch of `n` (query, page) pairs, score every
query against every page, and treat the diagonal as correct. Each pair gets `n-1` negatives for free.

Two details that matter more than they look:

* **Scale the scores before the cross-entropy.** MeanMaxSim lives in roughly `[-1, 1]`; softmax over
  values that small is nearly uniform and produces almost no gradient. Multiplying by a temperature
  (here `1/0.05 = 20`) is what makes it trainable. This is the same trick CLIP uses.
* **Batch size *is* the number of negatives.** A batch of 4 is a 4-way classification problem — far too
  easy. Real training runs use 64–256 with gradient checkpointing, or a cached loss (10.4) to decouple
  batch size from memory.

We deliberately run only a handful of steps here — the point is a working, inspectable loop, not a
state-of-the-art model.

In [ ]:
import copy
import random


def encode_for_training(queries, images):
    """Tokenize a training batch. Returns the two BatchFeatures, on device."""
    q = retriever_processor.apply_chat_template(
        [[{"role": "user", "content": text}] for text in queries],
        task="query", tokenize=True, return_dict=True, return_tensors="pt",
        processor_kwargs={"padding": "longest"},
    ).to(DEVICE)
    d = retriever_processor.apply_chat_template(
        [[{"role": "user", "content": [{"type": "image", "image": image}]}] for image in images],
        task="document", tokenize=True, return_dict=True, return_tensors="pt",
        processor_kwargs={"padding": "longest", "max_side": 640},   # smaller pages = trainable on a T4
    ).to(DEVICE)
    return q, d


def contrastive_loss(model, queries, images, temperature=0.05, alpha=0.5):
    """Joint late-interaction + dense InfoNCE, the objective NeoMME-Retriever was trained with."""
    q_inputs, d_inputs = encode_for_training(queries, images)
    q_out = model(**q_inputs)
    d_out = model(**d_inputs)

    late = mean_maxsim(q_out.embeddings.float(), d_out.embeddings.float(),
                       a_mask=q_inputs["attention_mask"], b_mask=d_inputs["attention_mask"])
    dense = q_out.dense_embeddings.float() @ d_out.dense_embeddings.float().T

    targets = torch.arange(len(queries), device=late.device)
    loss_late = torch.nn.functional.cross_entropy(late / temperature, targets)
    loss_dense = torch.nn.functional.cross_entropy(dense / temperature, targets)
    return alpha * loss_late + (1 - alpha) * loss_dense, late


def finetune(model, pairs, steps=12, batch_size=4, lr=1e-5, temperature=0.05, seed=0):
    """A minimal but real contrastive fine-tuning loop."""
    rng = random.Random(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=steps, pct_start=0.25)

    model.train()
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    history = []
    for step in range(steps):
        batch = rng.sample(pairs, min(batch_size, len(pairs)))
        queries = [text for text, _ in batch]
        images = [CORPUS[DOC_IDS.index(doc)] for _, doc in batch]

        loss, scores = contrastive_loss(model, queries, images, temperature=temperature)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        accuracy = float((scores.argmax(dim=1) == torch.arange(len(batch), device=scores.device))
                         .float().mean())
        history.append((float(loss), accuracy))
        print(f"  step {step + 1:>3}/{steps}  loss={float(loss):.4f}  "
              f"in-batch acc={accuracy:.2f}  lr={scheduler.get_last_lr()[0]:.2e}")

    model.eval()
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    return history


print("✅ training loop defined "
      "(run the next cell to actually train — it takes a couple of minutes)")

### 10.3 Train, and measure whether it helped

**Always evaluate on held-out pages.** In-batch accuracy going to 1.0 tells you nothing; with a batch of
4 it is a 4-way choice. The number that matters is retrieval quality over the whole corpus on pages the
model never saw a query for.

In [ ]:
RUN_TRAINING = True     # set False to skip (this cell fine-tunes all 260M parameters)

def evaluate_on(model, pairs, corpus=CORPUS, doc_ids=DOC_IDS):
    """Full-corpus retrieval metrics for a set of (query, gold_doc) pairs."""
    was_training = model.training
    model.eval()
    global retriever
    previous, retriever = retriever, model
    try:
        idx = encode_documents(corpus, doc_ids, batch_size=4)
        q_dense, q_multi = encode_queries([text for text, _ in pairs])
        probe = [Query(text=text, gold=doc, lang="?", kind="?") for text, doc in pairs]
        late_metrics, _ = evaluate(mean_maxsim(q_multi, idx.multi), probe, idx.doc_ids)
        dense_metrics, _ = evaluate(cos_sim(q_dense, idx.dense), probe, idx.doc_ids)
    finally:
        retriever = previous
        if was_training:
            model.train()
    return late_metrics, dense_metrics


if RUN_TRAINING:
    print("── before fine-tuning ───────────────────────────────────────────")
    before_late, before_dense = evaluate_on(retriever, EVAL_PAIRS)
    report("  held-out, late interaction", before_late)
    report("  held-out, dense", before_dense)

    tuned = copy.deepcopy(retriever)
    print("\n── training ─────────────────────────────────────────────────────")
    with Timer("fine-tuning"):
        history = finetune(tuned, TRAIN_PAIRS, steps=12, batch_size=4, lr=1e-5)

    print("\n── after fine-tuning ────────────────────────────────────────────")
    after_late, after_dense = evaluate_on(tuned, EVAL_PAIRS)
    report("  held-out, late interaction", after_late)
    report("  held-out, dense", after_dense)

    print("\n── delta ────────────────────────────────────────────────────────")
    for name, before, after in (("late interaction", before_late, after_late),
                                ("dense", before_dense, after_dense)):
        change = after["nDCG@5"] - before["nDCG@5"]
        arrow = "▲" if change > 0 else ("▼" if change < 0 else "=")
        print(f"  {name:<18} nDCG@5 {before['nDCG@5']:.3f} -> {after['nDCG@5']:.3f}  {arrow} {change:+.3f}")

    plt.figure(figsize=(7, 3.4))
    plt.plot([h[0] for h in history], "o-", label="loss")
    plt.plot([h[1] for h in history], "s--", label="in-batch accuracy")
    plt.xlabel("step"); plt.grid(alpha=0.3); plt.legend(); plt.title("Fine-tuning progress")
    plt.tight_layout(); plt.show()
else:
    print("skipped (RUN_TRAINING = False)")

> **Read that delta honestly.** Twelve training pairs over six pages, twelve optimizer steps. The metric
> may go *down* — with a sample this small that is noise, not evidence, and it is exactly why the loop
> prints held-out numbers rather than training loss. What you should take away is the mechanics: a working
> objective, a clean split, and a measurement that would tell you the truth on 10,000 pairs.
>
> **Realistic settings** for an actual run: 5k–50k pairs, batch size 64–256 (cached loss), `lr` 1e-5 to
> 5e-5 with warmup, 1–3 epochs, hard negatives mined from the base model's own top-50 mistakes.
>
> **One more thing this demo cuts.** We loaded the model in bf16/fp16 and are optimizing those weights
> directly. That is fine for a 12-step illustration and wrong for a real run — AdamW needs fp32 master
> weights or the updates disappear into rounding. Load with `dtype=torch.float32` and use `bf16=True`
> (mixed precision) in your training arguments, which keeps fp32 parameters and casts only the forward pass.

### 10.4 The Sentence Transformers route

For real training runs you want the Sentence Transformers stack: `MultiVectorEncoder` wraps the model,
`MultiVectorMultipleNegativesRankingLoss` is the loss from 10.2 done properly, and
`CachedMultiVectorMultipleNegativesRankingLoss` decouples the effective batch size from GPU memory via
GradCache — which is how you get 256 in-batch negatives on one card.

`MultiVectorEncoder` expects a checkpoint saved in the Sentence Transformers layout. The NeoMME retriever
checkpoints are published for use with Sentence Transformers, so this should load directly; the cell below
degrades gracefully and tells you what it found either way.

In [ ]:
try:
    from sentence_transformers import (
        MultiVectorEncoder,
        MultiVectorEncoderTrainer,
        MultiVectorEncoderTrainingArguments,
    )
    from sentence_transformers.multi_vector_encoder.losses import (
        CachedMultiVectorMultipleNegativesRankingLoss,
        MultiVectorMultipleNegativesRankingLoss,
    )
    from sentence_transformers.multi_vector_encoder.evaluation import (
        MultiVectorInformationRetrievalEvaluator,
    )

    st_model = MultiVectorEncoder(RETRIEVER_ID, device=DEVICE)
    print(f"✅ loaded {RETRIEVER_ID} as a MultiVectorEncoder")
    print(st_model)

    # encode_query / encode_document apply the right task prompt for you.
    q = st_model.encode_query(["Which quarter had the strongest coffee sales?"])
    d = st_model.encode_document([CORPUS[DOC_IDS.index("coffee_sales")]])
    print(f"\nencode_query    -> {[tuple(np.shape(x)) for x in q]}")
    print(f"encode_document -> {[tuple(np.shape(x)) for x in d]}")
    print(f"similarity      -> {st_model.similarity(q, d)}")
    ST_AVAILABLE = True

except Exception as exc:
    ST_AVAILABLE = False
    print(f"⚠️  Could not load via MultiVectorEncoder: {type(exc).__name__}: {exc}")
    print("\n   This is not fatal — Section 10.2's raw PyTorch loop trains the same model.")
    print("   Check the model card in the NeoMME collection for the current Sentence")
    print("   Transformers layout, and make sure sentence-transformers >= 6.0.0.")

Here is the training script you would run once that loads. It is left as a non-executing cell because a
real run wants a real dataset and a GPU you are not sharing with a tutorial.

In [ ]:
TRAINING_SCRIPT = r"""
from datasets import Dataset
from sentence_transformers import (
    MultiVectorEncoder, MultiVectorEncoderTrainer, MultiVectorEncoderTrainingArguments,
)
from sentence_transformers.multi_vector_encoder.losses import (
    CachedMultiVectorMultipleNegativesRankingLoss,
)
from sentence_transformers.multi_vector_encoder.evaluation import (
    MultiVectorInformationRetrievalEvaluator,
)

model = MultiVectorEncoder("Hcompany/NeoMME-260M-Retriever")

# Columns are (anchor, positive) — the trainer treats every other row as a negative.
# `image` columns hold PIL images; `mine_hard_negatives` can add a `negative` column.
train_dataset = Dataset.from_dict({"query": [...], "image": [...]})
eval_dataset = Dataset.from_dict({"query": [...], "image": [...]})

# GradCache: 256 in-batch negatives with the memory of a batch of 8.
loss = CachedMultiVectorMultipleNegativesRankingLoss(model, mini_batch_size=8)

args = MultiVectorEncoderTrainingArguments(
    output_dir="neomme-retriever-mydomain",
    num_train_epochs=2,
    per_device_train_batch_size=256,
    learning_rate=2e-5,
    warmup_ratio=0.05,
    bf16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    logging_steps=25,
)

evaluator = MultiVectorInformationRetrievalEvaluator(
    queries={...}, corpus={...}, relevant_docs={...}, name="my-domain-eval",
)

trainer = MultiVectorEncoderTrainer(
    model=model, args=args, train_dataset=train_dataset,
    eval_dataset=eval_dataset, loss=loss, evaluator=evaluator,
)
trainer.train()
model.save_pretrained("neomme-retriever-mydomain/final")
"""

print(TRAINING_SCRIPT)

### 10.5 Domain-adaptive continued pretraining

If your documents contain a vocabulary the model has genuinely never seen — chemical formulae, ICD-10
codes, a low-resource language — fine-tuning the retriever alone fights an uphill battle: the backbone has
no good representation to contrast in the first place.

The fix is to run more of the **original objective** on your unlabelled corpus first. You need no labels
at all, just pages and/or text.

In [ ]:
RECIPE = r"""
1. Collect unlabelled in-domain data — page images, raw text, or both.
   Text needs no annotation; images need no captions.

2. Continue masked-LM pretraining on the backbone:

   from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer

   model = AutoModelForMaskedLM.from_pretrained("Hcompany/NeoMME-260M")
   collator = DataCollatorForLanguageModeling(
       tokenizer=processor.tokenizer, mlm=True, mlm_probability=0.30,
   )
   # NeoMME pretrains with a *variable* mask ratio (discrete diffusion), so a
   # higher, and ideally randomized, mlm_probability is closer to the original
   # objective than BERT's fixed 15%.
   Trainer(model=model, args=..., train_dataset=..., data_collator=collator).train()
   model.save_pretrained("neomme-260m-mydomain")

3. Sanity-check with fill-mask (Section 4) on in-domain sentences. Domain terms
   should now be predicted where the base model guessed generic words.

4. Initialize NeoMMEForRetrieval from your adapted backbone and run Section 10.2/10.4:

   from transformers import NeoMMEForRetrieval
   retriever = NeoMMEForRetrieval.from_pretrained("neomme-260m-mydomain")
   # the two retrieval heads start from scratch, so give this stage more steps
   # and a slightly higher learning rate than a plain retriever fine-tune

Rule of thumb: skip step 2 unless fill-mask on your own text is visibly poor.
Continued pretraining is expensive and, on ordinary business documents, usually
buys less than a day spent on better training pairs.
"""
print(RECIPE)

<a name="s11"></a>
## 11. From retrieval to visual RAG

Retrieval returns a page. Users want an **answer**. The visual-RAG pattern closes that gap without an OCR
pipeline anywhere in it:

```
   query ──▶ NeoMME-Retriever ──▶ top-k page IMAGES ──▶ VLM ──▶ grounded answer + page citation
```

The contrast with classic RAG is the point. Classic RAG needs a document → text pipeline: OCR, layout
analysis, table reconstruction, chunking. Every stage loses information, and the stage that loses the most
is the one that flattens a bar chart into a caption. Here the retriever consumes pixels and the generator
consumes pixels; nothing is ever flattened to text until the model writes the answer.

### 11.1 The retrieval half — already done

`search()` from Section 8.7 is the whole retrieval side. Let's package the context assembly.

In [ ]:
def retrieve_context(question, top_k=2, rerank_depth=5):
    """Return the top-k pages for a question, ready to hand to a VLM."""
    hits = search(question, top_k=top_k, rerank_depth=rerank_depth, show=False)
    return [{"doc_id": doc_id, "score": score, "image": image} for doc_id, score, image in hits]


question = "What is the total amount due on the Northwind invoice, and when do I have to pay it?"
context = retrieve_context(question, top_k=2)

print(f"❓ {question}\n")
for rank, hit in enumerate(context, 1):
    print(f"  {rank}. {hit['doc_id']}  (score {hit['score']:.3f})")
show_images([h["image"] for h in context],
            [f"#{r} {h['doc_id']}" for r, h in enumerate(context, 1)], cols=2, width=3.6)

### 11.2 The generation half

NeoMME is an **encoder** — it has no decoder and cannot write an answer. That is by design: it is a
retrieval model, and pairing it with a small generator is far cheaper than making one model do both jobs.

Any vision-language model works. Below we use a small open VLM; swap `VLM_ID` for whatever you have
access to (a local model, or an API-served one). The cell is optional and skips cleanly if the model can't
be loaded — everything before it still works.

In [ ]:
RUN_VLM = True                                  # set False to skip the download
VLM_ID = "Qwen/Qwen2.5-VL-3B-Instruct"          # any VLM with a chat template works

vlm = vlm_processor = None
if RUN_VLM:
    try:
        from transformers import AutoModelForImageTextToText, AutoProcessor as VLMAutoProcessor

        vlm_processor = VLMAutoProcessor.from_pretrained(VLM_ID)
        vlm = AutoModelForImageTextToText.from_pretrained(
            VLM_ID, dtype=DTYPE, device_map="auto",
        ).eval()
        print(f"✅ loaded generator: {VLM_ID} ({param_count(vlm):,} params)")
    except Exception as exc:
        print(f"⚠️  Could not load {VLM_ID}: {type(exc).__name__}: {exc}")
        print("   Section 11.3 will fall back to showing the retrieved evidence only.")
else:
    print("skipped (RUN_VLM = False)")

In [ ]:
SYSTEM_RULES = (
    "You answer questions about document pages. Use only what is visible in the provided "
    "page images. Quote the exact figures you see. If the pages do not contain the answer, "
    "say so plainly instead of guessing."
)


def generate_answer(question, context, max_new_tokens=160):
    """Ask the VLM to answer using the retrieved page images as the only evidence."""
    if vlm is None:
        return None

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_RULES}]},
        {"role": "user", "content": (
            [{"type": "image", "image": hit["image"]} for hit in context]
            + [{"type": "text", "text": question}]
        )},
    ]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(vlm.device)

    with torch.inference_mode():
        generated = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = generated[0, inputs["input_ids"].shape[1]:]
    return vlm_processor.decode(new_tokens, skip_special_tokens=True).strip()


def visual_rag(question, top_k=2, show=True):
    """Full pipeline: retrieve page images with NeoMME, answer with the VLM."""
    context = retrieve_context(question, top_k=top_k)
    answer = generate_answer(question, context)

    print(f"\n{'=' * 78}\n❓ {question}\n{'-' * 78}")
    print("📄 retrieved:")
    for rank, hit in enumerate(context, 1):
        print(f"   {rank}. {hit['doc_id']}  (score {hit['score']:.3f})")
    if answer is None:
        gold = next(s["fact"] for s in CORPUS_SPECS if s["id"] == context[0]["doc_id"])
        print(f"\n💬 (no generator loaded — the top page states: {gold})")
    else:
        print(f"\n💬 {answer}")
    if show:
        show_images([h["image"] for h in context],
                    [h["doc_id"] for h in context], cols=len(context), width=3.4)
    return context, answer


for question in [
    "What is the total amount due on the Northwind invoice, and when do I have to pay it?",
    "Which hour of the day had the highest electricity generation, and how many GW was it?",
    "How many weeks of paid parental leave do employees get, and can they extend it?",
]:
    visual_rag(question, top_k=2, show=False)

### 11.3 A cross-lingual question, end to end

The retriever finds a Spanish page from an English question, and the generator answers in English by
reading the Spanish page image. No translation step exists anywhere in this pipeline.

In [ ]:
visual_rag("What is the speed limit for forklifts in the warehouse?", top_k=1)

### 11.4 What to watch out for

| Failure | Symptom | Fix |
|---|---|---|
| Retrieval misses | Confident answer from the wrong page | Increase `top_k`; check Section 8.6's per-query ranks; fine-tune (Section 10) |
| Resolution too low for the VLM | "I cannot read the figure" | Pass the **original** page image to the VLM, not the resized copy you indexed |
| Context blowup | OOM in `generate` | `top_k=1`–3 pages; a VLM's per-image token cost is usually higher than NeoMME's |
| Hallucinated citations | Figures that appear on no page | Keep the "use only what is visible" system rule; show the pages next to the answer |
| Latency | Slow end to end | Cache the document index; only the query encode and the generation are per-request |

The most important line in that table is the second one. **Index at a resolution tuned for retrieval;
generate from the highest-resolution copy you have.** They are separate budgets and there is no reason to
make them the same.

In [ ]:
if vlm is not None:
    del vlm, vlm_processor
    vlm = vlm_processor = None
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("✅ generator released")

<a name="s12"></a>
## 12. Production notes

### 12.1 Throughput: measure it on your own pages

The published figure is ~51 pages/s for the 260M model at 2048×2048 on an L40S. Your number will differ.
Here is the measurement, so you can run it on your hardware at your resolution.

In [ ]:
@torch.inference_mode()
def benchmark(image, batch_size=4, repeats=3, warmup=1, **image_kwargs):
    messages = [[{"role": "user", "content": [{"type": "image", "image": image}]}]] * batch_size
    inputs = retriever_processor.apply_chat_template(
        messages, task="document", tokenize=True, return_dict=True, return_tensors="pt",
        processor_kwargs={"padding": "longest", **image_kwargs},
    ).to(DEVICE)

    for _ in range(warmup):
        retriever(**inputs)

    times = []
    for _ in range(repeats):
        with Timer() as t:
            retriever(**inputs)
        times.append(t.seconds)

    seq_len = inputs["input_ids"].shape[1]
    return batch_size / min(times), seq_len


page = CORPUS[0]
print(f"device: {DEVICE}  dtype: {DTYPE}\n")
print(f"{'resize setting':<26}{'seq len':>10}{'pages/s':>11}{'ms/page':>10}")
print("-" * 57)
for label, kwargs in [
    ("max_side=512", {"max_side": 512}),
    ("max_side=768", {"max_side": 768}),
    ("max_side=1024", {"max_side": 1024}),
    ("native (900x1200)", {}),
]:
    try:
        rate, seq_len = benchmark(page, batch_size=4, **kwargs)
        print(f"{label:<26}{seq_len:>10,}{rate:>11.1f}{1000 / rate:>10.1f}")
    except RuntimeError as exc:                       # OOM on small GPUs
        print(f"{label:<26}{'—':>10}{'OOM':>11}{'':>10}")
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

### 12.2 The knobs that actually move the needle

| Knob | Effect | Notes |
|---|---|---|
| `max_side` / `max_pixels` | **Quadratic** on compute | The single biggest lever. Find the smallest value that keeps your text legible. |
| `dtype=torch.bfloat16` | ~2× throughput, ~half the memory | Use fp16 on pre-Ampere cards; fp32 only for training. |
| `attn_implementation="flash_attention_2"` | Large win at long sequence lengths | Needs the `flash-attn` package and a compatible GPU. |
| batch size | Better GPU utilization | Group pages of **similar size** — padding to the longest in the batch is pure waste. |
| `output_multivector=False` | Skips the multi-vector head | Use it for a dense-only index. |
| `output_dense=False` | Skips the dense head | Rarely worth it — the dense head is a mean-pool. |
| `dense_dim=N` | Smaller dense index | Free at encode time; costs nothing to try. |

In [ ]:
LOADING_PATTERNS = r"""
# ── Throughput-first inference ────────────────────────────────────────────────
retriever = NeoMMEForRetrieval.from_pretrained(
    "Hcompany/NeoMME-260M-Retriever",
    dtype=torch.bfloat16,                     # fp16 on pre-Ampere
    attn_implementation="flash_attention_2",  # needs `pip install flash-attn`
    device_map="auto",
).eval()

# ── Dense-only indexing (smallest index, fastest sweep) ───────────────────────
outputs = retriever(**inputs, output_multivector=False, dense_dim=256)

# ── Length-sorted batching: sort pages by patch count before batching, so each ─
#    batch pads to nearly nothing.
def sorted_batches(images, processor, batch_size):
    sizes = [processor.image_processor.get_number_of_image_patches(im.height, im.width)
             for im in images]
    order = sorted(range(len(images)), key=lambda i: sizes[i])
    for start in range(0, len(order), batch_size):
        picks = order[start:start + batch_size]
        yield picks, [images[i] for i in picks]
    # remember to un-sort the results back into the original order

# ── Memory-bounded scoring for very large candidate sets ──────────────────────
scores = mean_maxsim(query_vectors, doc_vectors, chunk_elements=50_000_000)
"""
print(LOADING_PATTERNS)

### 12.3 Storing the index

**Dense vectors** are ordinary embeddings — FAISS, pgvector, Qdrant, Milvus, LanceDB all handle them.
Use `dense_dim` to pick a width and `quantize_embeddings` to pick a precision, then treat it like any
other 256- to 1024-dimensional index.

**Multi-vector representations** need a store that understands late interaction. Three options, in
increasing order of effort:

1. **Rerank only.** Never index the multi-vectors at all — keep them in object storage keyed by page ID,
   fetch the few hundred candidates the dense stage returned, and score in memory. This is the two-stage
   pipeline from 8.5 and it is what most teams should do.
2. **A vector DB with native multi-vector support.** Qdrant and Vespa both score MaxSim server-side.
3. **PLAID-style indexing.** Cluster all token vectors, store centroid IDs plus residuals, and do
   approximate MaxSim. Highest performance, most machinery.

In [ ]:
INDEX_SKETCH = r"""
# Sketch of the recommended two-stage layout.
#
#   Postgres/pgvector   : page_id -> dense int8 vector (256-d)         ~256 B/page
#   Object storage (S3) : page_id -> pooled int8 multi-vectors         ~35 KB/page at pool_factor=4
#
#   query:
#     1. q_dense, q_multi = encode_query(text)              # one forward pass
#     2. candidates = pgvector.knn(q_dense, k=200)          # milliseconds
#     3. blobs = s3.mget([c.page_id for c in candidates])   # parallel fetch
#     4. scores = mean_maxsim([q_multi], [dequantize(b) for b in blobs])
#     5. return top 10 by score
#
# Step 2 is the only part that has to scale with corpus size; steps 3-4 scale with k.
# Both encoders come from the same forward pass in step 1 — there is no second model.
"""
print(INDEX_SKETCH)

### 12.4 A checklist before you ship

- [ ] **Pin your resolution.** Decide `max_side` by measuring recall, not by intuition, and record it —
      re-indexing later because you changed it is expensive.
- [ ] **Index and generate at different resolutions.** See 11.4.
- [ ] **Sort by size before batching.** Padding to the longest page in a mixed batch can double your bill.
- [ ] **Keep queries in float32.** Asymmetric quantization is nearly free precisely because you only
      degrade the document side.
- [ ] **Store the model revision with the index.** Embeddings from two checkpoints are not comparable, and
      the failure is silent.
- [ ] **Build a labelled eval set early.** Two hundred real queries with gold pages will tell you more
      than any benchmark, and Section 8.2's twenty lines of metrics are all the tooling you need.
- [ ] **Watch for pages that produce very long sequences.** A 16k context is generous, but a poster-sized
      scan at native resolution will exceed it — cap `max_side` or split the page.

<a name="s13"></a>
## 13. Troubleshooting and FAQ

### 13.1 Errors you are likely to hit

**`KeyError: 'neomme'` or `ImportError: cannot import name 'NeoMMEProcessor'`**
Your `transformers` predates the model. Re-run the install cell (`transformers` from `main`) and restart
the runtime.

**`ValueError: The NeoMME retrieval template must preserve its leading task marker.`**
Truncation removed the `<query>` or `<doc>` marker. Raise `max_length` in `processor_kwargs`, or stop
truncating.

**`ValueError: Truncation removed NeoMME query expansion tokens; increase max_length.`**
Same cause, different casualty — the ten trailing `<mask>` tokens got cut. Your `max_length` must fit the
query text *plus* 11 marker tokens.

**`ValueError: NeoMME image inputs contain an invalid or truncated token layout.`**
The `<doc><img>` + grid + `<row>` layout was disturbed. You almost certainly truncated an image input,
or hand-built `input_ids`. Let `apply_chat_template` build them.

**`ValueError: NeoMME image batches require padding to a common sequence length.`**
Add `processor_kwargs={"padding": "longest"}` (or `padding=True` on a direct `processor(...)` call).

**`ValueError: NeoMME cannot encode text and images in the same conversation.`**
One conversation, one modality. Encode a caption as its own document.

**`ValueError: NeoMME image content must use task='document'.`**
Images are documents. To search *with* an image you need a different setup than this template provides.

**`ValueError: Got N image patches for M image placeholder tokens`**
`pixel_values` and `input_ids` came from different calls. Always use the single processor call's output
as one unit.

**`ValueError: dense_dim must be in 1..1024 (the pooled width)`**
`dense_dim` cannot exceed `config.hidden_size`.

**`RuntimeError: CUDA out of memory` during indexing**
In order: lower the document batch size, then `max_side`, then switch to `dtype=torch.bfloat16`, then
`output_multivector=False` if you only need a dense index.

**Retrieval quality is bad, and it was fine in this notebook**
Ninety percent of the time it is one of two things: (a) you encoded queries with `task="document"`, or
(b) `max_side` is small enough that the text on your pages is no longer legible. Check (a) by printing
`convert_ids_to_tokens` on a query — the first token must be `<query>`. Check (b) by saving a resized page
and trying to read it yourself.

### 13.2 FAQ

**Is NeoMME a generative model?**
No. It is an encoder with no decoder, and it cannot write text. `NeoMMEForMaskedLM` predicts masked
*tokens* — useful for probing and continued pretraining, not for generation. Pair it with a VLM for
answers (Section 11).

**Do I need OCR?**
No. That is the point. The model reads pixels.

**How many images can I put in one forward pass?**
One image per *conversation*; as many conversations per *batch* as memory allows. The 16,384-token context
holds roughly two 4K pages, but the retrieval template deliberately restricts you to one image per
document, because a page is the right unit of retrieval.

**Dense or late interaction?**
Both, from one forward pass. Dense to sweep, late interaction to rerank (Section 8.5). Use dense alone
only when storage is the binding constraint and you have measured the quality cost.

**Which languages does it support?**
It is multilingual by construction — a 131k-token vocabulary trained from scratch on multilingual data.
Section 8.6 measures cross-lingual retrieval across six European languages. Test *your* languages with
Section 4's fill-mask probe before committing.

**Can I use it for text-only retrieval?**
Yes — encode text documents with `task="document"`. You lose the model's main differentiator, though, so
compare against a dedicated text embedder on your data.

**What about the `predecay` checkpoints?**
`Hcompany/NeoMME-{260M,800M}-Pretrain-predecay-s450000` are snapshots taken *before* the final
learning-rate decay. They are for researchers who want to branch their own annealing or continued
pretraining. For applications, use the released backbone or retriever.

**Licensing?**
Every checkpoint is Apache 2.0, including commercial use.

### 13.3 Where to go next

* **[The NeoMME blog post](https://huggingface.co/blog/Hcompany/neomme)** — the announcement, with
  benchmarks and design rationale.
* **[The NeoMME collection](https://huggingface.co/collections/Hcompany/neomme)** — all checkpoints and
  their model cards. Check these for the current recommended settings.
* **[Transformers `NeoMME` docs](https://huggingface.co/docs/transformers/main/model_doc/neomme)** —
  the authoritative API reference for `NeoMMEConfig`, `NeoMMEProcessor`, `NeoMMEImageProcessor`,
  `NeoMMEModel`, `NeoMMEForMaskedLM` and `NeoMMEForRetrieval`.
* **[Sentence Transformers multi-vector quickstart](https://sbert.net/docs/quickstart.html#multi-vector-encoder)** —
  `MultiVectorEncoder`, the losses, the evaluators, and hard-negative mining.
* **[ViDoRe](https://huggingface.co/vidore)** — the visual document retrieval benchmark NeoMME reports on.
  Evaluate your fine-tune here before believing your own numbers.

### 13.4 What this notebook covered

| Section | You can now… |
|---|---|
| 3 | Encode text, images, or both through one bidirectional encoder, and read the config that explains why it is fast |
| 4 | Probe the model with fill-mask, in six languages, and score text with pseudo-log-likelihood |
| 5 | Predict and control the token cost of any page before you pay it |
| 6 | Build a labelled multilingual document corpus from scratch |
| 7 | Format queries and documents correctly, and score them two different ways |
| 8 | Build a two-stage search engine and measure it with real IR metrics |
| 9 | Shrink the index by one to two orders of magnitude, and know what it cost you |
| 10 | Fine-tune on your own pairs — in raw PyTorch or Sentence Transformers |
| 11 | Turn retrieved pages into grounded answers with a VLM |
| 12 | Ship it |

---

*Built against `transformers` `main` (NeoMME contributed 2026-08-31) and `sentence-transformers >= 6.0.0`.
All checkpoints Apache 2.0. If an API has moved since, the model card in the
[NeoMME collection](https://huggingface.co/collections/Hcompany/neomme) is the source of truth.*